# Xenium – 03 Both runs: cell-type composition, T/B-border inference and Tg positioning

**Input (from the 02 notebooks):**

| object | file | content |
|---|---|---|
| `adata_run1_all` | `ALLcells_run1.h5ad` | run 1, all annotated cells |
| `adata_run2_all` | `20261019_ALLcells_run2.h5ad` | run 2, all annotated cells |
| `ad_run1_tb` | `20260417_TBcells_subannotated.h5ad` | run 1, Tg + T + B cells |
| `ad_run2_tb` | `20260520_TBcells_run2.h5ad` | run 2, Tg + T + B cells |

Excluded section: `6:Day3_0388` (QC; already removed in notebook 02, run 1).

Sections
1. Setup and data
2. Cell-type composition (all Tg cells, all cell types incl. FDCs, per sample)
3. Spatial maps and annotation-marker QC
4. T/B-border inference (main parameters + parameter scan)
5. Tg cells relative to the T/B border (counts, fraction at the border)
6. Tg density: border vs deep T and deep B zone
7. ECDF of the signed distance to the border (per section, mean ± SEM) and position summary

**Figure panels produced here** (output files carry the panel as prefix)

| panel | section | output |
|---|---|---|
| Fig. 4b (top), Suppl. Fig. 4a | 3 | `SuppFig4a_Fig4b_<run>_spatial_celltypes.pdf/.png` |
| Fig. 4b (bottom) | 4 | `Fig4b_TB_border_<run>_<section>_px20_s1.5_h75.png` |
| Fig. 4c | 2.2 | `Fig4c_Mean_cell_fractions_per_day_FDC.pdf` |
| Fig. 4d | 3 | `Fig4d_Dotplot_annotation_markers_<section>.pdf` |
| Fig. 4e | 5 | `Fig4e_Tg_fraction_at_TB_border_per_subset_bars.pdf` |
| Fig. 4f | 7 | `Fig4f_Tg_ECDF_per_LN_mean_sem.pdf` |
| Fig. 4g | 7 | `Fig4g_Tg_border_position_median_per_LN.pdf` |
| Suppl. Fig. 4b | 3 | `LN_umap_heatmaps/SuppFig4b_UMAP_celltypes_<section>.pdf` |

Fig. 4h (gene heatmap across the border) and Fig. 4i (expression profiles across the border) are not yet part of this repository.
All other plots are QC / supporting plots that are not in the manuscript.

## 1. Setup and data

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
# ------------------------------------------------------------
# Paths relative to this folder (adapt BASE_DIR if your data live elsewhere)
# ------------------------------------------------------------
BASE_DIR = ".."
RAW_DIR  = f"{BASE_DIR}/01_Raw"
DATA_DIR = f"{BASE_DIR}/02_ProcessedData"

RUN1_ALL = f"{DATA_DIR}/ALLcells_run1.h5ad"
RUN2_ALL = f"{DATA_DIR}/20261019_ALLcells_run2.h5ad"
RUN1_TB  = f"{DATA_DIR}/20260417_TBcells_subannotated.h5ad"
RUN2_TB  = f"{DATA_DIR}/20260520_TBcells_run2.h5ad"

EXCLUDE_SECTIONS = ["6:Day3_0388"]   # excluded after QC (already removed in notebook 02)

In [ ]:
def load(path):
    a = sc.read(path)
    a = a[~a.obs["timepoint"].astype(str).isin(EXCLUDE_SECTIONS)].copy()
    if hasattr(a.obs["timepoint"], "cat"):
        a.obs["timepoint"] = a.obs["timepoint"].cat.remove_unused_categories()
    return a

adata_run1_all = load(RUN1_ALL)
adata_run2_all = load(RUN2_ALL)
ad_run1_tb = load(RUN1_TB)
ad_run2_tb = load(RUN2_TB)

for name, a in [("run1 all", adata_run1_all), ("run2 all", adata_run2_all),
                ("run1 Tg+T+B", ad_run1_tb), ("run2 Tg+T+B", ad_run2_tb)]:
    print(f"{name:12s} {a.n_obs:>8d} cells, sections: {sorted(a.obs['timepoint'].astype(str).unique())}")

Colours and marker genes (as in the 02 notebooks).

In [ ]:
ln_major_colors = {
    "B_cells": "#4A90E2",        # soft blue
    "T_cells": "#50E3C2",        # gentle green
    "Tg_T_cells": "#D0021B",     # magenta / highlight
    "Macrophages": "#F5A623",    # warm gold
    "Stromal_cells": "#9B9B9B",  # neutral gray-violet
    "Lymphatic_EC": "#2E8B57",    # teal-cyan,
    "Tregs": "#7B6FD0",          # muted violet
    "cDC1": "#084594",           # deep navy blue (clearly blue)
    "cDC2": "#6B8E23",           # olive / yellow-green (clearly green)
    "GC_B_cells": "#6A3D9A",      # deep purple
    "Neutrophils": "#636363",
    "NA": "white"
}

In [ ]:
ln_major_markers2  = {
    "Tregs": ["Cd3d","Cd4", "Foxp3", "Il2ra"],
    "cDC1": ["Xcr1", "Itgax", "Cd74", "Irf8"],
    "cDC2": ["Sirpa", "Itgax", "Cd74", "Il6"],
    #"GC_B_cells": ["Cd79a", "Cxcr5", "Aicda", "Bcl6"],
    "GC_B_cells": ["Aicda"],
    "B_cells": ["Cd79a","Cr2", "Cd38", "Cxcr5"],
    "T_cells": ["Cd3d", "Ccr7", "Cd4", "Cd8a"],
    "Tg_T_cells": ["ENA_Y11102_Y11102.1", "ENA_Y11103_Y11103.1", "CD45.1", "Cd3d", "Cd4"],
    "Macrophages": ["Cd68", "Sirpa"],
    "Stromal_cells": ["Timp3", "Cfh", "Cd21", "Cxcl13", "Ccl21"],
    "Lymphatic_EC": ["Lyve1", "Prox1", "Plvap"],
    "Neutrophils" :["Lst1", "Csf3r", "Pglyrp1", "Mmp9"]
}

## 2. Cell-type composition

### 2.1 All Tg cells per day (both runs, before any border filtering)

In [ ]:
# ============================================================
# STACKED BARPLOT: ALL Tg CELL POPULATIONS PER DAY
# RUN1 + RUN2 COMBINED
#
# IMPORTANT:
# This analysis is performed DIRECTLY from the original
# AnnData objects, before any T–B border / corridor inference.
#
# Therefore:
#   - no dist_to_border filtering
#   - no tb_analysis_keep filtering
#   - no border-zone filtering
#
# Input objects:
#   ad_run1_tb, ad_run2_tb (Tg + T + B objects of both runs)
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================


# ============================================================
# 1. INPUT OBJECTS
# ============================================================

ad_run1 = ad_run1_tb
ad_run2 = ad_run2_tb

runs = {
    "run1": ad_run1,
    "run2": ad_run2
}


# ============================================================
# 2. SETTINGS
# ============================================================

module_palette = {
    "Tg_pre_Tfh": "#6baed6",
    "Tg_Tfh": "#08519c",
    "Tg_pre_Th1": "#fdae6b",
    "Tg_Th1": "#a63603"
}


tg_types = [
    "Tg_pre_Th1",
    "Tg_Th1",
    "Tg_pre_Tfh",
    "Tg_Tfh"
]


day_order = [
    0,
    2,
    3,
    4,
    6
]


# ============================================================
# 3. INSPECT ORIGINAL .obs COLUMNS
# ============================================================

for run_name, ad in runs.items():

    print(
        f"\n================ {run_name} ================"
    )

    print(
        "Number of cells:",
        ad.n_obs
    )

    print(
        "\n.obs columns:"
    )

    print(
        ad.obs.columns.tolist()
    )


# ============================================================
# 4. USE ORIGINAL TIMEPOINT COLUMN
#
# In your T–B border workflow, LN is created from:
#
#   adata.obs["timepoint"]
#
# Therefore we use timepoint directly here.
# ============================================================

TIMEPOINT_COL = "timepoint"


for run_name, ad in runs.items():

    if TIMEPOINT_COL not in ad.obs.columns:

        raise ValueError(
            f"{TIMEPOINT_COL!r} not found in "
            f"{run_name}.obs.\n\n"
            f"Available columns:\n"
            f"{ad.obs.columns.tolist()}"
        )


# ============================================================
# 5. IDENTIFY THE Tg ANNOTATION COLUMN
#
# We do not assume that the original AnnData uses
# the name 'fine_type'.
#
# Search common annotation-column names and choose the one
# containing the Tg labels.
# ============================================================

candidate_annotation_columns = [
    "compartment_annotation_2",
    "fine_type",

    "Celltype_2",
    "celltype",
    "cell_type",
    "annotation",
    "Annotation",
    "predicted.Celltype_2",
    "predicted.id",
    "type"
]


def find_tg_annotation_column(adata, tg_types):

    # --------------------------------------------------------
    # First search likely annotation column names
    # --------------------------------------------------------

    for column in candidate_annotation_columns:

        if column not in adata.obs.columns:
            continue

        values = set(
            adata.obs[column]
            .dropna()
            .astype(str)
            .unique()
        )

        overlap = set(tg_types).intersection(
            values
        )

        if len(overlap) > 0:

            print(
                f"Found Tg annotation column: "
                f"{column}"
            )

            print(
                "Matching Tg labels:",
                sorted(overlap)
            )

            return column


    # --------------------------------------------------------
    # If none of the common names worked, search ALL
    # categorical/string obs columns
    # --------------------------------------------------------

    for column in adata.obs.columns:

        series = adata.obs[column]

        if (
            isinstance(
                series.dtype,
                pd.CategoricalDtype
            )
            or series.dtype == object
        ):

            values = set(
                series
                .dropna()
                .astype(str)
                .unique()
            )

            overlap = set(
                tg_types
            ).intersection(
                values
            )

            if len(overlap) > 0:

                print(
                    f"Found Tg labels in column: "
                    f"{column}"
                )

                print(
                    "Matching Tg labels:",
                    sorted(overlap)
                )

                return column


    return None


annotation_columns = {}


for run_name, ad in runs.items():

    annotation_col = (
        find_tg_annotation_column(
            ad,
            tg_types
        )
    )

    if annotation_col is None:

        raise ValueError(
            f"\nCould not find a column containing the "
            f"Tg labels in {run_name}.\n\n"
            f"Available columns:\n"
            f"{ad.obs.columns.tolist()}"
        )

    annotation_columns[
        run_name
    ] = annotation_col


print(
    "\nAnnotation columns used:"
)

print(
    annotation_columns
)


# ============================================================
# 6. COMBINE METADATA FROM ORIGINAL OBJECTS
#
# IMPORTANT:
# We are copying directly from ad.obs here.
# Nothing from the T–B border analysis is used.
# ============================================================

obs_list = []


for run_name, ad in runs.items():

    annotation_col = (
        annotation_columns[
            run_name
        ]
    )


    obs_tmp = pd.DataFrame({

        "cell_id":
            ad.obs_names.astype(str),

        "run":
            run_name,

        "timepoint":
            ad.obs[
                TIMEPOINT_COL
            ].astype(str).values,

        "fine_type":
            ad.obs[
                annotation_col
            ].astype(str).values

    })


    obs_list.append(
        obs_tmp
    )


tg_all = pd.concat(
    obs_list,
    axis=0,
    ignore_index=True
)


print(
    "\nTotal cells after combining original objects:",
    len(tg_all)
)


print(
    "\nCells per run:"
)

print(
    tg_all[
        "run"
    ].value_counts()
)


# ============================================================
# 7. EXTRACT DAY FROM ORIGINAL timepoint
#
# Handles strings such as:
#
#   10:Day0_0382
#   Day2_0386
#   4:Day3_0388
# ============================================================

tg_all["day"] = (
    tg_all[
        "timepoint"
    ]
    .astype(str)
    .str.extract(
        r"Day(\d+)",
        expand=False
    )
)


tg_all["day"] = pd.to_numeric(
    tg_all["day"],
    errors="coerce"
)


print(
    "\nDetected days:"
)

print(
    sorted(
        tg_all[
            "day"
        ]
        .dropna()
        .unique()
    )
)


# ============================================================
# 8. SANITY CHECK THE Tg LABELS
# ============================================================

print(
    "\nFine-type labels containing 'Tg':"
)


tg_labels_found = sorted(
    [
        x
        for x in tg_all[
            "fine_type"
        ]
        .dropna()
        .unique()
        if "Tg" in str(x)
    ]
)


print(
    tg_labels_found
)


# ============================================================
# 9. KEEP ONLY Tg POPULATIONS
#
# Again:
# NO distance filtering.
# NO corridor filtering.
# ============================================================

tg_all_tg = tg_all.loc[
    tg_all[
        "fine_type"
    ].isin(
        tg_types
    )
    &
    tg_all[
        "day"
    ].isin(
        day_order
    )
].copy()


print(
    "\n================================"
)

print(
    "ALL Tg CELLS USED FOR BARPLOT:"
)

print(
    len(tg_all_tg)
)

print(
    "================================"
)


# ============================================================
# 10. SANITY CHECK COUNTS BY RUN
# ============================================================

print(
    "\nTg cells per run:"
)


display(
    tg_all_tg
    .groupby(
        [
            "run",
            "fine_type"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="n_cells"
    )
)


# ============================================================
# 11. SANITY CHECK COUNTS BY RUN × DAY
# ============================================================

print(
    "\nTg cells per run × day:"
)


display(
    tg_all_tg
    .groupby(
        [
            "run",
            "day"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="n_cells"
    )
)


# ============================================================
# 12. COUNT CELLS PER DAY × Tg SUBTYPE
# ============================================================

tg_counts_per_day = (
    tg_all_tg
    .groupby(
        [
            "day",
            "fine_type"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="n_cells"
    )
)


# ============================================================
# 13. ADD MISSING COMBINATIONS AS ZERO
# ============================================================

full_index = (
    pd.MultiIndex
    .from_product(
        [
            day_order,
            tg_types
        ],
        names=[
            "day",
            "fine_type"
        ]
    )
)


tg_counts_per_day = (
    tg_counts_per_day
    .set_index(
        [
            "day",
            "fine_type"
        ]
    )
    .reindex(
        full_index,
        fill_value=0
    )
    .reset_index()
)


print(
    "\nCombined run1 + run2 counts:"
)


display(
    tg_counts_per_day
)


# ============================================================
# 14. PIVOT FOR STACKED BARPLOT
# ============================================================

plot_df = (
    tg_counts_per_day
    .pivot(
        index="day",
        columns="fine_type",
        values="n_cells"
    )
    .reindex(
        index=day_order,
        columns=tg_types,
        fill_value=0
    )
)


display(
    plot_df
)


# ============================================================
# 15. COLORS
# ============================================================

plot_colors = [
    module_palette[
        celltype
    ]
    for celltype
    in plot_df.columns
]


# ============================================================
# 16. STACKED BARPLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 4.5)
)


plot_df.plot(
    kind="bar",
    stacked=True,
    width=0.70,
    color=plot_colors,
    ax=ax
)


ax.set_xlabel(
    "Day",
    fontsize=12
)


ax.set_ylabel(
    "Number of Tg cells",
    fontsize=12
)


ax.set_title(
    "Tg cell populations per day\n"
    "(all cells, run1 + run2 combined)",
    fontsize=14
)


ax.set_xticklabels(
    [
        f"d{int(day)}"
        for day
        in day_order
    ],
    rotation=0
)


ax.legend(
    title="Tg subtype",
    bbox_to_anchor=(
        1.02,
        1
    ),
    loc="upper left",
    frameon=False
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


plt.tight_layout()


fig.savefig(
    "Tg_cell_populations_ALL_cells_run1_run2_stacked.pdf",
    bbox_inches="tight",
    dpi=300
)


plt.show()

### 2.2 Mean cell-type composition per day (incl. FDCs) → Figure 4c

Stromal cells in the top 25 % of an FDC module score (Cxcl13, Cd21/Cr2), computed per run, are
re-labelled as FDCs (`compartment_annotation_FDC`). Proportions are calculated per sample and averaged per day.

In [ ]:
# ============================================================
# Mean stacked barplot of cell-type proportions per day
# across two Xenium runs
#
# FDC RE-ANNOTATION:
#
# 1. Subset cells annotated as "Stromal_cells"
# 2. Calculate an FDC module score from:
#       Cxcl13
#       Cd21 / Cr2
# 3. Within each Xenium run, select the TOP 25% of stromal
#    cells by FDC module score
# 4. Re-annotate those cells as "FDCs"
# 5. Calculate cell-type proportions separately per sample
# 6. Combine samples from both runs
# 7. Average sample-level proportions per day
# 8. Plot mean proportions as a stacked barplot
#
# IMPORTANT:
# The original annotation column is NOT overwritten.
# A new column called "compartment_annotation_FDC" is created.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================


# ============================================================
# 1. SETTINGS
# ============================================================

# Work on copies so the original objects remain unchanged
adata_run1 = adata_run1_all.copy()
adata_run2 = adata_run2_all.copy()


# Sample column
sample_col = "timepoint"


# Original superclass annotation
original_celltype_col = "compartment_annotation_2"


# New annotation containing the FDC subdivision
celltype_col = "compartment_annotation_FDC"


# Existing stromal label
stromal_label = "Stromal_cells"


# New FDC label
fdc_label = "FDCs"


# Fraction of stromal cells classified as FDCs
FDC_TOP_FRACTION = 0.25


day_order = [
    "Day0",
    "Day2",
    "Day3",
    "Day4",
    "Day6"
]


# Order in stacked barplot
superclass_order = [
    "B_cells",
    "GC_B_cells",
    "Lymphatic_EC",
    "Macrophages",
    "Neutrophils",
    "Stromal_cells",
    "FDCs",
    "T_cells",
    "Tg_T_cells",
    "Tregs",
    "cDC1",
    "cDC2"
]


# ============================================================
# 2. HELPER FUNCTION:
# EXTRACT EXPRESSION OF ONE GENE
# ============================================================

def get_gene_expression(
    adata,
    gene
):
    """
    Extract expression values for one gene from AnnData.
    Works for dense and sparse expression matrices.
    """

    if gene not in adata.var_names:

        raise ValueError(
            f"{gene} not found in adata.var_names."
        )


    values = adata[
        :,
        gene
    ].X


    if hasattr(
        values,
        "toarray"
    ):

        values = values.toarray()


    values = np.asarray(
        values
    ).flatten()


    return values


# ============================================================
# 3. HELPER FUNCTION:
# SAFE Z-SCORE
# ============================================================

def safe_zscore(values):
    """
    Z-score expression values.

    If there is no variation, return zeros instead of NaN.
    """

    values = np.asarray(
        values,
        dtype=float
    )


    mean_value = np.nanmean(
        values
    )


    sd_value = np.nanstd(
        values,
        ddof=0
    )


    if (
        not np.isfinite(sd_value)
        or sd_value == 0
    ):

        return np.zeros(
            len(values),
            dtype=float
        )


    return (
        values
        - mean_value
    ) / sd_value


# ============================================================
# 4. FUNCTION:
# CALCULATE FDC MODULE SCORE AND RE-ANNOTATE TOP 25%
# ============================================================

def annotate_fdcs_by_module_score(
    adata,
    run_name,
    original_celltype_col,
    stromal_label="Stromal_cells",
    new_annotation_col="compartment_annotation_FDC",
    top_fraction=0.25
):
    """
    Re-annotate the highest-scoring stromal cells as FDCs.

    FDC module score:
        mean(
            z-score(Cxcl13),
            z-score(Cd21/Cr2)
        )

    Z-scoring is performed ONLY across stromal cells.

    The top `top_fraction` of stromal cells within this run
    is assigned as FDCs.
    """


    # --------------------------------------------------------
    # Check annotation column
    # --------------------------------------------------------

    if original_celltype_col not in adata.obs.columns:

        raise ValueError(
            f"{original_celltype_col} not found in "
            f"{run_name}.obs."
        )


    # --------------------------------------------------------
    # Check Cxcl13
    # --------------------------------------------------------

    if "Cxcl13" not in adata.var_names:

        raise ValueError(
            f"Cxcl13 not found in {run_name}."
        )


    # --------------------------------------------------------
    # Identify CD21 gene
    #
    # Mouse CD21 is normally Cr2.
    # --------------------------------------------------------

    if "Cd21" in adata.var_names:

        cd21_gene = "Cd21"

    elif "Cr2" in adata.var_names:

        cd21_gene = "Cr2"

    else:

        raise ValueError(
            f"Neither Cd21 nor Cr2 was found in {run_name}."
        )


    print(
        "\n=========================================="
    )

    print(
        f"FDC module-score annotation: {run_name}"
    )

    print(
        f"Markers: Cxcl13 + {cd21_gene}"
    )

    print(
        "=========================================="
    )


    # --------------------------------------------------------
    # Create new annotation column
    # --------------------------------------------------------

    adata.obs[
        new_annotation_col
    ] = (
        adata.obs[
            original_celltype_col
        ]
        .astype(str)
        .copy()
    )


    # --------------------------------------------------------
    # Identify stromal cells
    # --------------------------------------------------------

    stromal_mask = (
        adata.obs[
            original_celltype_col
        ]
        .astype(str)
        == stromal_label
    ).to_numpy()


    stromal_indices = np.where(
        stromal_mask
    )[0]


    n_stromal = len(
        stromal_indices
    )


    print(
        "Number of stromal cells:",
        n_stromal
    )


    if n_stromal == 0:

        print(
            "No stromal cells found. "
            "Skipping FDC annotation."
        )

        adata.obs[
            "FDC_module_score"
        ] = np.nan

        adata.obs[
            "FDC_top25"
        ] = False

        return adata, cd21_gene


    # --------------------------------------------------------
    # Extract expression across ALL cells
    # --------------------------------------------------------

    cxcl13_all = get_gene_expression(
        adata,
        "Cxcl13"
    )


    cd21_all = get_gene_expression(
        adata,
        cd21_gene
    )


    # --------------------------------------------------------
    # Restrict expression to stromal cells
    # --------------------------------------------------------

    cxcl13_stromal = (
        cxcl13_all[
            stromal_indices
        ]
    )


    cd21_stromal = (
        cd21_all[
            stromal_indices
        ]
    )


    # --------------------------------------------------------
    # Z-score each marker across stromal cells
    # --------------------------------------------------------

    cxcl13_z = safe_zscore(
        cxcl13_stromal
    )


    cd21_z = safe_zscore(
        cd21_stromal
    )


    # --------------------------------------------------------
    # FDC module score
    #
    # Equal contribution of the two markers
    # --------------------------------------------------------

    fdc_score_stromal = (
        cxcl13_z
        + cd21_z
    ) / 2


    # --------------------------------------------------------
    # Determine top-25% threshold
    # --------------------------------------------------------

    threshold = np.quantile(
        fdc_score_stromal,
        1 - top_fraction
    )


    print(
        "Top-25% FDC score threshold:",
        round(
            threshold,
            4
        )
    )


    # --------------------------------------------------------
    # Select top-scoring stromal cells
    # --------------------------------------------------------

    fdc_stromal_mask = (
        fdc_score_stromal
        >= threshold
    )


    # --------------------------------------------------------
    # Store scores for ALL cells
    # --------------------------------------------------------

    adata.obs[
        "FDC_module_score"
    ] = np.nan


    adata.obs.iloc[
        stromal_indices,
        adata.obs.columns.get_loc(
            "FDC_module_score"
        )
    ] = fdc_score_stromal


    # --------------------------------------------------------
    # Store logical FDC assignment
    # --------------------------------------------------------

    adata.obs[
        "FDC_top25"
    ] = False


    fdc_cell_indices = (
        stromal_indices[
            fdc_stromal_mask
        ]
    )


    adata.obs.iloc[
        fdc_cell_indices,
        adata.obs.columns.get_loc(
            "FDC_top25"
        )
    ] = True


    # --------------------------------------------------------
    # Re-annotate selected cells
    # --------------------------------------------------------

    adata.obs.iloc[
        fdc_cell_indices,
        adata.obs.columns.get_loc(
            new_annotation_col
        )
    ] = fdc_label


    # --------------------------------------------------------
    # QC
    # --------------------------------------------------------

    n_fdcs = len(
        fdc_cell_indices
    )


    print(
        "Cells annotated as FDCs:",
        n_fdcs
    )


    print(
        "Fraction of stromal cells annotated as FDCs:",
        round(
            100
            * n_fdcs
            / n_stromal,
            2
        ),
        "%"
    )


    print(
        "\nFDC module-score summary:"
    )


    display(
        pd.DataFrame({
            "Cxcl13": cxcl13_stromal,
            cd21_gene: cd21_stromal,
            "Cxcl13_z": cxcl13_z,
            f"{cd21_gene}_z": cd21_z,
            "FDC_module_score": fdc_score_stromal,
            "FDC": fdc_stromal_mask
        })
        .describe()
    )


    return (
        adata,
        cd21_gene
    )


# ============================================================
# 5. ANNOTATE FDCs IN RUN 1
# ============================================================

adata_run1, cd21_gene_run1 = (
    annotate_fdcs_by_module_score(
        adata=adata_run1,
        run_name="Run1",
        original_celltype_col=original_celltype_col,
        stromal_label=stromal_label,
        new_annotation_col=celltype_col,
        top_fraction=FDC_TOP_FRACTION
    )
)


# ============================================================
# 6. ANNOTATE FDCs IN RUN 2
# ============================================================

adata_run2, cd21_gene_run2 = (
    annotate_fdcs_by_module_score(
        adata=adata_run2,
        run_name="Run2",
        original_celltype_col=original_celltype_col,
        stromal_label=stromal_label,
        new_annotation_col=celltype_col,
        top_fraction=FDC_TOP_FRACTION
    )
)


# ============================================================
# 7. QC:
# NUMBER OF FDCs PER SAMPLE
# ============================================================

def summarize_fdcs_per_sample(
    adata,
    run_name,
    sample_col
):

    tmp = adata.obs[
        [
            sample_col,
            original_celltype_col,
            celltype_col,
            "FDC_module_score",
            "FDC_top25"
        ]
    ].copy()


    tmp[
        sample_col
    ] = (
        tmp[
            sample_col
        ]
        .astype(str)
    )


    # Restrict QC to stromal cells
    tmp = tmp.loc[
        tmp[
            original_celltype_col
        ]
        .astype(str)
        == stromal_label
    ].copy()


    qc = (
        tmp
        .groupby(
            sample_col,
            observed=True
        )
        .agg(

            n_stromal=(
                original_celltype_col,
                "size"
            ),

            n_FDC=(
                "FDC_top25",
                "sum"
            ),

            mean_FDC_score=(
                "FDC_module_score",
                "mean"
            )

        )
        .reset_index()
    )


    qc[
        "FDC_fraction_of_stromal"
    ] = (
        qc[
            "n_FDC"
        ]
        / qc[
            "n_stromal"
        ]
    )


    qc[
        "run"
    ] = run_name


    qc[
        "day"
    ] = (
        qc[
            sample_col
        ]
        .str.extract(
            r"(Day\d+)",
            expand=False
        )
    )


    return qc


fdc_qc_run1 = (
    summarize_fdcs_per_sample(
        adata_run1,
        "Run1",
        sample_col
    )
)


fdc_qc_run2 = (
    summarize_fdcs_per_sample(
        adata_run2,
        "Run2",
        sample_col
    )
)


fdc_qc_all = pd.concat(
    [
        fdc_qc_run1,
        fdc_qc_run2
    ],
    ignore_index=True
)


print(
    "\nFDC QC per sample:"
)


display(
    fdc_qc_all
    .sort_values(
        [
            "day",
            "run",
            sample_col
        ]
    )
)


# ============================================================
# 8. FUNCTION TO CALCULATE SAMPLE-LEVEL FRACTIONS
# ============================================================

def calculate_sample_fractions(
    adata,
    run_name,
    sample_col,
    celltype_col
):
    """
    Calculate cell-type proportions independently for every
    Xenium sample.

    Each resulting row corresponds to one biological sample.
    Missing cell types are represented by zero.
    """


    obs = adata.obs[
        [
            sample_col,
            celltype_col
        ]
    ].copy()


    # Remove missing values
    obs = obs.dropna(
        subset=[
            sample_col,
            celltype_col
        ]
    )


    obs[
        sample_col
    ] = (
        obs[
            sample_col
        ]
        .astype(str)
    )


    obs[
        celltype_col
    ] = (
        obs[
            celltype_col
        ]
        .astype(str)
    )


    # --------------------------------------------------------
    # Sample × cell-type proportions
    # --------------------------------------------------------

    fractions = pd.crosstab(
        index=obs[
            sample_col
        ],
        columns=obs[
            celltype_col
        ],
        normalize="index"
    )


    fractions[
        "sample"
    ] = (
        fractions.index.astype(str)
    )


    fractions[
        "run"
    ] = run_name


    fractions[
        "day"
    ] = (
        fractions[
            "sample"
        ]
        .str.extract(
            r"(Day\d+)",
            expand=False
        )
    )


    fractions = fractions.reset_index(
        drop=True
    )


    return fractions


# ============================================================
# 9. CALCULATE SAMPLE-LEVEL PROPORTIONS FOR BOTH RUNS
# ============================================================

fractions_run1 = (
    calculate_sample_fractions(
        adata=adata_run1,
        run_name="Run1",
        sample_col=sample_col,
        celltype_col=celltype_col
    )
)


fractions_run2 = (
    calculate_sample_fractions(
        adata=adata_run2,
        run_name="Run2",
        sample_col=sample_col,
        celltype_col=celltype_col
    )
)


# ============================================================
# 10. COMBINE BOTH XENIUM RUNS
# ============================================================

fractions_all = pd.concat(
    [
        fractions_run1,
        fractions_run2
    ],
    ignore_index=True,
    sort=False
)


# ============================================================
# 11. IDENTIFY PROPORTION COLUMNS
# ============================================================

metadata_cols = [
    "sample",
    "run",
    "day"
]


fraction_cols = [
    col
    for col in fractions_all.columns
    if col not in metadata_cols
]


fractions_all[
    fraction_cols
] = (
    fractions_all[
        fraction_cols
    ]
    .fillna(
        0
    )
)


# ============================================================
# 12. QUALITY-CONTROL CHECKS
# ============================================================

print(
    "\nDetected samples:"
)


display(
    fractions_all[
        [
            "run",
            "sample",
            "day"
        ]
    ]
    .sort_values(
        [
            "day",
            "run",
            "sample"
        ]
    )
)


# ------------------------------------------------------------
# Proportions should sum to one
# ------------------------------------------------------------

fractions_all[
    "proportion_sum"
] = (
    fractions_all[
        fraction_cols
    ]
    .sum(
        axis=1
    )
)


print(
    "\nProportion sums per sample:"
)


display(
    fractions_all[
        [
            "run",
            "sample",
            "day",
            "proportion_sum"
        ]
    ]
    .sort_values(
        [
            "day",
            "run",
            "sample"
        ]
    )
)


assert (
    fractions_all[
        "proportion_sum"
    ]
    .between(
        0.9999,
        1.0001
    )
    .all()
), (
    "At least one sample does not sum to 1."
)


# ============================================================
# 13. NUMBER OF BIOLOGICAL SAMPLES PER DAY
# ============================================================

samples_per_day = (
    fractions_all
    .groupby(
        "day",
        observed=True
    )
    .size()
    .reindex(
        day_order
    )
)


print(
    "\nNumber of samples per day:"
)


display(
    samples_per_day
    .rename(
        "n_samples"
    )
    .to_frame()
)


# ============================================================
# 14. AVERAGE SAMPLE-LEVEL PROPORTIONS PER DAY
# ============================================================

mean_day_fractions = (
    fractions_all
    .groupby(
        "day",
        observed=True
    )[
        fraction_cols
    ]
    .mean()
    .reindex(
        day_order
    )
)


print(
    "\nMean proportion sums per day:"
)


display(
    mean_day_fractions
    .sum(
        axis=1
    )
    .rename(
        "proportion_sum"
    )
    .to_frame()
)


# ============================================================
# 15. REORDER STACKED CATEGORIES
# ============================================================

present_superclasses = [
    celltype
    for celltype in superclass_order
    if celltype
    in mean_day_fractions.columns
]


additional_superclasses = [
    celltype
    for celltype
    in mean_day_fractions.columns
    if celltype
    not in superclass_order
]


plot_order = (
    present_superclasses
    + additional_superclasses
)


mean_day_fractions = (
    mean_day_fractions
    .reindex(
        columns=plot_order
    )
)


# ============================================================
# 16. PALETTE
# ============================================================

# Copy existing palette so the original is not changed
plot_palette = (
    ln_major_colors.copy()
)


# Add one color for FDCs
plot_palette[
    "FDCs"
] = "#8c6bb1"


missing_colors = [
    celltype
    for celltype
    in mean_day_fractions.columns
    if celltype
    not in plot_palette
]


print(
    "\nCategories missing from palette:"
)

print(
    missing_colors
)


plot_colors = [
    plot_palette.get(
        celltype,
        "lightgrey"
    )
    for celltype
    in mean_day_fractions.columns
]


# ============================================================
# 17. PLOT MEAN STACKED PROPORTIONS
# ============================================================

fig, ax = plt.subplots(figsize=(4.5, 3))

mean_day_fractions.mul(100).plot(
    kind="bar",
    stacked=True,
    width=0.5,
    color=plot_colors,
    linewidth=0,
    ax=ax
)

ax.set_title("Mean Cell Fractions per Day Across Xenium Runs")
ax.set_xlabel("Day")
ax.set_ylabel("Mean cell fraction (%)")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0)

ax.legend(
    title="Superclass",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ============================================================
# 18. SAVE AS PDF
# ============================================================

fig.savefig(
    "Fig4c_Mean_cell_fractions_per_day_FDC.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

plt.show()


# ============================================================
# 19. NUMERIC TABLE USED FOR PLOTTING
# ============================================================

mean_percentage_table = (
    mean_day_fractions
    .mul(
        100
    )
    .round(
        2
    )
)


print(
    "\nMean percentages used in the plot:"
)


display(
    mean_percentage_table
)


# ============================================================
# 20. INDIVIDUAL SAMPLE PROPORTIONS
# ============================================================

sample_percentage_table = (
    fractions_all[
        [
            "run",
            "sample",
            "day"
        ]
        + plot_order
    ]
    .copy()
)


sample_percentage_table[
    plot_order
] = (
    sample_percentage_table[
        plot_order
    ]
    * 100
)


sample_percentage_table[
    plot_order
] = (
    sample_percentage_table[
        plot_order
    ]
    .round(
        2
    )
)


print(
    "\nIndividual sample percentages:"
)


display(
    sample_percentage_table
    .sort_values(
        [
            "day",
            "run",
            "sample"
        ]
    )
)

### 2.3 Composition per sample, grouped by day

In [ ]:
# ============================================================
# 21. STACKED BARPLOT PER SAMPLE (grouped by day)
#
# Same data, colors and category order as the mean plot,
# but one bar per lymph node / sample instead of the day mean.
# Requires: fractions_all, plot_order, plot_colors, day_order
# ============================================================

GAP = 0.8                 # extra space between days
BAR_W = 0.8
SHOW_MEAN_BAR = False     # True = add the day-mean bar (hatched) after each day's samples
FS = 8

plot_df = fractions_all.copy()
plot_df["day"] = pd.Categorical(plot_df["day"], categories=day_order, ordered=True)

# LN number from sample id, e.g. "10:Day0_0382" -> 10
plot_df["ln_num"] = plot_df["sample"].str.extract(r"^(\d+):", expand=False).astype(float)
plot_df["label"] = "LN" + plot_df["ln_num"].astype("Int64").astype(str)
plot_df = plot_df.sort_values(["day", "run", "ln_num"]).reset_index(drop=True)

# x positions with a gap between days
x_pos, day_centers, x = [], {}, 0.0
bars = []          # (x, values, label, is_mean)
for day in day_order:
    sub = plot_df[plot_df["day"] == day]
    if sub.empty:
        continue
    start = x
    for _, r in sub.iterrows():
        bars.append((x, r[plot_order].values.astype(float), r["label"], False))
        x += 1
    if SHOW_MEAN_BAR:
        bars.append((x, sub[plot_order].mean().values.astype(float), "mean", True))
        x += 1
    day_centers[day] = (start + x - 1) / 2
    x += GAP

with plt.rc_context({"font.size": FS, "axes.linewidth": 0.7}):
    fig, ax = plt.subplots(figsize=(0.28 * len(bars) + 2.6, 3))

    for xb, vals, lab, is_mean in bars:
        bottom = 0.0
        for v, c in zip(vals * 100, plot_colors):
            ax.bar(xb, v, bottom=bottom, width=BAR_W, color=c, linewidth=0,
                   hatch="////" if is_mean else None, edgecolor="white" if is_mean else None)
            bottom += v

    ax.set_xticks([b[0] for b in bars])
    ax.set_xticklabels([b[2] for b in bars], rotation=90, fontsize=FS - 1)
    ax.set_xlim(-0.7, x - GAP - 0.3)
    ax.set_ylim(0, 100)
    ax.set_ylabel("Cell fraction (%)")
    ax.tick_params(length=2, width=0.7)

    # day labels below the sample labels
    for day, xc in day_centers.items():
        ax.text(xc, -0.20, day.replace("Day", "d"), transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=FS, fontweight="bold")

    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in plot_colors]
    ax.legend(handles[::-1], [l.replace("_", " ") for l in plot_order][::-1],
              title="Superclass", bbox_to_anchor=(1.01, 1), loc="upper left",
              frameon=False, fontsize=FS - 1, title_fontsize=FS, handlelength=1, handleheight=1)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.savefig("Cell_Fractions_Per_Sample_Xenium_Runs_FDC_top25_module_score.pdf",
                bbox_inches="tight", pad_inches=0.02, transparent=True)
    plt.show()

## 3. Spatial maps and annotation-marker QC

**Supplementary Figure 4a** (all sections of both runs) and **Figure 4b, top** (one section shown):
cell-type annotation in space, Tg T cells drawn on top.

In [ ]:
from matplotlib.lines import Line2D

# ============================================================
# SETTINGS
# ============================================================

celltype_col = "compartment_annotation_FDC"

plot_palette = ln_major_colors.copy()
plot_palette["FDCs"] = "#8c6bb1"

tg_color = "#E31A1C"
point_size = 0.5


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_spatial(adata, run_name):

    labels = adata.obs[celltype_col].astype(str)
    coords = adata.obsm["spatial"]

    # Identify Tg populations
    tg_labels = [
        label for label in labels.unique()
        if "tg" in label.lower()
    ]

    if not tg_labels:
        raise ValueError(
            f"No Tg populations found: {labels.unique()}"
        )

    is_tg = labels.isin(tg_labels).to_numpy()

    # Use original colors, with Tg populations in red
    colors = np.array([
        tg_color if label in tg_labels
        else plot_palette.get(label, "lightgrey")
        for label in labels
    ])

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(10, 10))

    # First: all non-Tg cells
    ax.scatter(
        coords[~is_tg, 0],
        coords[~is_tg, 1],
        c=colors[~is_tg].tolist(),
        s=point_size,
        marker="o",
        edgecolors="none",
        linewidths=0,
        alpha=1,
        zorder=1,
        rasterized=False
    )

    # Second: Tg cells ON TOP
    ax.scatter(
        coords[is_tg, 0],
        coords[is_tg, 1],
        c=tg_color,
        s=point_size,
        marker="o",
        edgecolors="none",
        linewidths=0,
        alpha=1,
        zorder=10,
        rasterized=False
    )

    # Xenium spatial orientation
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")

    ax.set_title(
        f"{run_name} – Updated FDC annotation"
    )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    categories = adata.obs[
        celltype_col
    ].astype("category").cat.categories

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            markerfacecolor=(
                tg_color if c in tg_labels
                else plot_palette.get(c, "lightgrey")
            ),
            markeredgecolor="none",
            markersize=6,
            label=c
        )
        for c in categories
    ]

    ax.legend(
        handles=legend_handles,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    # --------------------------------------------------------
    # High-resolution export
    # --------------------------------------------------------

    fig.savefig(
        f"SuppFig4a_Fig4b_{run_name}_spatial_celltypes.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        f"SuppFig4a_Fig4b_{run_name}_spatial_celltypes.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(
        f"{run_name}: {is_tg.sum():,} Tg cells drawn last"
    )


# ============================================================
# RUN 1
# ============================================================

plot_spatial(adata_run1, "Run1")


# ============================================================
# RUN 2
# ============================================================

plot_spatial(adata_run2, "Run2")

### Marker heatmap and dot plot of the annotation markers (one sample)

Dot plot → **Figure 4d** (section `3:Day4_0391`). The heatmap is a QC plot (not in the manuscript).

In [ ]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
adata_src = adata_run1                       # AnnData of the run
sample_col = "timepoint"                     # obs column with the sample / LN id
sample_sel = "3:Day4_0391"                   # the one sample to plot
celltype_col = "compartment_annotation_FDC"  # annotation column

show_groups = True    # False = no cell-type brackets above the genes
CELL_W = 0.15         # inches per gene column
CELL_H = 0.075         # inches per cell-type row      # inches per heatmap cell (smaller = more compact)
FS = 10                # font size in pt at final size

gene_rename = {       # shorter labels for long gene names (adjust)
    "ENA_Y11102_Y11102.1": "Tg TCRα",
    "ENA_Y11103_Y11103.1": "Tg TCRβ",
}

# ------------------------------------------------------------
# 1. Subset to the one sample
# ------------------------------------------------------------
ad_plot = adata_src[adata_src.obs[sample_col] == sample_sel].copy()
print(f"{sample_sel}: {ad_plot.n_obs} cells")
if ad_plot.n_obs == 0:
    raise ValueError(f"No cells for {sample_sel}. Available: {sorted(adata_src.obs[sample_col].unique())}")

# ------------------------------------------------------------
# 2. Marker genes: present in data, each gene only once
# ------------------------------------------------------------
markers_plot, used, missing = {}, set(), []
for ct, genes in ln_major_markers2.items():
    keep = []
    for g in genes:
        if g not in ad_plot.var_names:
            missing.append(g)
        elif g not in used:
            keep.append(g)
            used.add(g)
    if keep:
        markers_plot[ct] = keep
print("Not in data:", sorted(set(missing)))

# ------------------------------------------------------------
# 3. Cell-type order as in the marker dict, nicer labels
# ------------------------------------------------------------
present_types = list(ad_plot.obs[celltype_col].astype(str).unique())
ct_order = [ct for ct in ln_major_markers2 if ct in present_types]
ct_order += [ct for ct in present_types if ct not in ct_order]

ad_plot.obs[celltype_col] = (
    ad_plot.obs[celltype_col].astype(str)
    .str.replace("_", " ")
    .astype("category")
    .cat.reorder_categories([ct.replace("_", " ") for ct in ct_order])
)

# rename long gene names
ad_plot.var.index = [gene_rename.get(g, g) for g in ad_plot.var_names]
ad_plot.var_names_make_unique()
markers_short = {
    ct.replace("_", " "): [gene_rename.get(g, g) for g in gs]
    for ct, gs in markers_plot.items()
}

n_genes = sum(len(g) for g in markers_short.values())
n_types = ad_plot.obs[celltype_col].nunique()

# ------------------------------------------------------------
# 4. Matrix plot
# ------------------------------------------------------------
with plt.rc_context({
    "font.size": FS,
    "xtick.labelsize": FS,
    "ytick.labelsize": FS,
    "axes.linewidth": 0.6,
}):
    mp = sc.pl.matrixplot(
        ad_plot,
        var_names=markers_short if show_groups else [g for gs in markers_short.values() for g in gs],
        groupby=celltype_col,
        standard_scale="var",
        cmap="Reds",
        colorbar_title="Scaled\nexpression",
        figsize=(CELL_W * n_genes + 1.3, CELL_H * n_types + (0.9 if show_groups else 0.4)),
        return_fig=True,
    )
    mp.style(edge_color="white", edge_lw=0.4)
    mp.legend(width=0.9)

    safe_id = sample_sel.replace(":", "_")
    mp.savefig(f"Heatmap_annotation_markers_{safe_id}.pdf", bbox_inches="tight", pad_inches=0.02)

In [ ]:
# ------------------------------------------------------------
# 4. Dot plot (Seurat style)
#    size  = % cells expressing (expression > 0)
#    color = average expression, z-scaled per gene across cell types (clipped ±2.5)
# ------------------------------------------------------------
from matplotlib.colors import LinearSegmentedColormap

DOT_W = 0.15          # inches per cell-type column
DOT_H = 0.1          # inches per gene row
MAX_DOT = 70          # dot area (pt²) for 100 % expressing
FS = 10

# genes in marker-dict order, first gene on top
genes_plot = list(dict.fromkeys(g for gs in markers_short.values() for g in gs))
types_plot = list(ad_plot.obs[celltype_col].cat.categories)

expr = ad_plot[:, genes_plot].X
expr = expr.toarray() if hasattr(expr, "toarray") else np.asarray(expr)
expr_df = pd.DataFrame(expr, columns=genes_plot, index=ad_plot.obs_names)
grp = ad_plot.obs[celltype_col].values

# Seurat-like: mean in linear space, back to log, then z-score per gene
avg = np.log1p(np.expm1(expr_df).groupby(grp).mean()).reindex(types_plot)
pct = (expr_df > 0).groupby(grp).mean().reindex(types_plot) * 100
avg_z = ((avg - avg.mean()) / avg.std(ddof=0).replace(0, np.nan)).clip(-2.5, 2.5).fillna(0)

long = (
    avg_z.stack().rename("avg_z").to_frame()
    .join(pct.stack().rename("pct"))
    .reset_index()
    .rename(columns={"level_0": "celltype", "level_1": "gene"})
)
long["x"] = long["celltype"].map({ct: i for i, ct in enumerate(types_plot)})
long["y"] = long["gene"].map({g: i for i, g in enumerate(reversed(genes_plot))})

cmap = LinearSegmentedColormap.from_list("seurat_bwr", ["#6f5cff", "#ffffff", "#ee2e2e"])

with plt.rc_context({"font.size": FS, "axes.linewidth": 0.6}):
    fig_w = DOT_W * len(types_plot) + 2.0
    fig_h = DOT_H * len(genes_plot) + 1.4
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sc_pts = ax.scatter(
        long["x"], long["y"],
        s=long["pct"] / 100 * MAX_DOT,
        c=long["avg_z"], cmap=cmap, vmin=-2.5, vmax=2.5,
        edgecolors="none",
    )

    ax.set_xticks(range(len(types_plot)))
    ax.set_xticklabels(types_plot, rotation=45, ha="right", rotation_mode="anchor", fontsize=FS)
    ax.set_yticks(range(len(genes_plot)))
    ax.set_yticklabels(list(reversed(genes_plot)), fontsize=FS, style="italic")
    ax.set_xlim(-0.6, len(types_plot) - 0.4)
    ax.set_ylim(-0.6, len(genes_plot) - 0.4)
    ax.tick_params(length=2, width=0.6, pad=1.5)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # legend: percent expressed
    size_handles = [
        plt.scatter([], [], s=p / 100 * MAX_DOT, color="black", edgecolors="none") for p in (25, 50, 75)
    ]
    leg = ax.legend(
        size_handles, ["25", "50", "75"], title="Percent\nexpressed",
        loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False,
        fontsize=FS - 1, title_fontsize=FS, labelspacing=0.6, handletextpad=0.4, borderaxespad=0,
    )
    leg._legend_box.align = "left"

    # colorbar: average expression
    cax = fig.add_axes([ax.get_position().x1 + 0.06, ax.get_position().y0 + 0.05,
                        0.025, ax.get_position().height * 0.35])
    cb = fig.colorbar(sc_pts, cax=cax, extend="both")
    cb.set_label("Average\nexpression", fontsize=FS, labelpad=3)
    cb.ax.tick_params(labelsize=FS - 1, length=2, width=0.6)
    cb.outline.set_linewidth(0.6)

    safe_id = sample_sel.replace(":", "_")
    fig.savefig(f"Fig4d_Dotplot_annotation_markers_{safe_id}.pdf", bbox_inches="tight", pad_inches=0.02)
    plt.show()

### UMAP and clustered heatmap of all panel genes for every section

UMAP → **Supplementary Figure 4b** (one section shown); the per-section heatmaps are QC (not in the manuscript).

In [ ]:
import os

# ============================================================
# UMAP + hclust heatmap of all panel genes for EVERY lymph node
# (loop over both Xenium runs)
#
# Heatmap is exported tall enough that every gene name is readable
# at FS_GENE pt when the PDF is viewed at 100 %.
# ============================================================

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
runs = {"run1": adata_run1, "run2": adata_run2}
sample_col = "timepoint"
celltype_col = "compartment_annotation_FDC"

RECOMPUTE_UMAP = True     # True = UMAP per LN; False = reuse existing X_umap
FS = 8                    # general font size (pt)
FS_GENE = 6               # gene label font size (pt)
ROW_H = 0.105             # inches per gene row (must be >= FS_GENE/72 ≈ 0.083 for no overlap)
COL_W = 0.22              # inches per cell-type column
MIN_CELLS = 50            # skip LNs with fewer cells

out_dir = "LN_umap_heatmaps"
os.makedirs(out_dir, exist_ok=True)

control_prefixes = ("NegControl", "BLANK", "Unassigned", "Deprecated", "antisense", "Intergenic")


def plot_one_ln(ad_src, run_label, sample_sel):
    safe_id = f"{run_label}_{sample_sel.replace(':', '_')}"

    # --------------------------------------------------------
    # 0. Subset
    # --------------------------------------------------------
    ad_ln = ad_src[ad_src.obs[sample_col].astype(str) == sample_sel].copy()
    if ad_ln.n_obs < MIN_CELLS:
        print(f"Skip {safe_id}: only {ad_ln.n_obs} cells")
        return None

    panel_genes = [g for g in ad_ln.var_names if not g.startswith(control_prefixes)]
    ad_ln = ad_ln[:, panel_genes].copy()
    ad_ln.obs[celltype_col] = ad_ln.obs[celltype_col].astype(str).str.replace("_", " ").astype("category")
    print(f"{safe_id}: {ad_ln.n_obs} cells, {ad_ln.n_vars} genes")

    # --------------------------------------------------------
    # 1. UMAP
    # --------------------------------------------------------
    if RECOMPUTE_UMAP or "X_umap" not in ad_ln.obsm:
        sc.pp.pca(ad_ln, n_comps=30)
        sc.pp.neighbors(ad_ln, n_neighbors=15, n_pcs=30)
        sc.tl.umap(ad_ln, random_state=0)

    with plt.rc_context({"font.size": FS, "axes.linewidth": 0.6}):
        fig_u, ax_u = plt.subplots(figsize=(3.6, 3))
        sc.pl.umap(ad_ln, color=celltype_col, ax=ax_u, show=False, frameon=False,
                   title=sample_sel, size=4, legend_fontsize=FS)
        for coll in ax_u.collections:
            coll.set_rasterized(True)
        fig_u.savefig(f"{out_dir}/SuppFig4b_UMAP_celltypes_{safe_id}.pdf", bbox_inches="tight", pad_inches=0.02, dpi=600)
        plt.close(fig_u)

    # --------------------------------------------------------
    # 2. Heatmap: mean per cell type, z-score per gene, hclust
    # --------------------------------------------------------
    expr = ad_ln.X.toarray() if hasattr(ad_ln.X, "toarray") else np.asarray(ad_ln.X)
    mean_df = (
        pd.DataFrame(expr, index=ad_ln.obs_names, columns=ad_ln.var_names)
        .groupby(ad_ln.obs[celltype_col].values)
        .mean()
        .T
    )
    mean_df = mean_df.loc[mean_df.std(axis=1) > 0]
    n_genes, n_types = mean_df.shape

    # size so that every gene label fits
    hm_h = ROW_H * n_genes
    hm_w = COL_W * n_types
    fig_w = hm_w + 2.2           # room for dendrogram + gene labels
    fig_h = hm_h + 2.0           # room for column dendrogram + cell-type labels

    with plt.rc_context({"font.size": FS, "axes.linewidth": 0.6}):
        g = sns.clustermap(
            mean_df,
            z_score=0,
            method="average",
            metric="correlation",
            row_cluster=True,
            col_cluster=True,
            cmap="vlag",
            center=0,
            vmin=-2.5, vmax=2.5,
            figsize=(fig_w, fig_h),
            dendrogram_ratio=(0.6 / fig_w, 0.9 / fig_h),   # fixed physical size in inches
            yticklabels=True,
            xticklabels=True,
            cbar_pos=(0.02, 1 - 0.8 / fig_h, 0.1 / fig_w, 0.6 / fig_h),
            cbar_kws={"label": "z-score", "extend": "both"},
            linewidths=0,
            rasterized=True,
        )
        ax = g.ax_heatmap
        ax.set_xlabel("")
        ax.set_ylabel("")
        g.fig.suptitle(f"{sample_sel} ({run_label}) – {n_genes} panel genes", fontsize=FS, y=1 + 0.15 / fig_h)
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=FS)
        plt.setp(ax.get_yticklabels(), fontsize=FS_GENE, style="italic")
        ax.tick_params(length=1, width=0.4, pad=1)
        g.cax.tick_params(labelsize=FS - 1, length=2)
        g.cax.yaxis.label.set_size(FS)

        g.savefig(f"{out_dir}/Heatmap_all_panel_genes_hclust_{safe_id}.pdf",
                  bbox_inches="tight", pad_inches=0.05, dpi=300)
        gene_order = mean_df.index[g.dendrogram_row.reordered_ind].tolist()
        plt.close(g.fig)

    pd.Series(gene_order, name="gene").to_csv(f"{out_dir}/Heatmap_gene_order_{safe_id}.csv", index=False)
    return {"run": run_label, "sample": sample_sel, "n_cells": ad_ln.n_obs,
            "n_genes": n_genes, "n_celltypes": n_types}


# ------------------------------------------------------------
# Loop over all LNs of both runs
# ------------------------------------------------------------
summary = []
for run_label, ad_src in runs.items():
    for sample_sel in sorted(ad_src.obs[sample_col].astype(str).unique()):
        res = plot_one_ln(ad_src, run_label, sample_sel)
        if res is not None:
            summary.append(res)

summary_df = pd.DataFrame(summary)
display(summary_df)
summary_df.to_csv(f"{out_dir}/summary_LN_plots.csv", index=False)
print(f"All plots saved in ./{out_dir}/")

## 4. T/B-border inference

Per section (Tg + T + B cells):
1. Endogenous T and B cells counted on a grid (main: 20 µm pixels); tissue mask from smoothed occupancy
2. Mask-aware Gaussian smoothing (main: σ = 1.5 px) of T and B counts; T–B score (T − B)/(T + B)
3. Border = zero isoline of the score (marching squares); contours < 100 vertices discarded
4. Contours split into 100-µm segments; segment QC: < 3 points, < 75 µm to tissue edge/hole,
   curvature > 90th percentile, hairpins (point within 100 µm that is ≥ 200 µm away along the contour);
   neighbouring segments of failed segments also excluded; ≥ 5 cells within 150 µm on each side
5. Every cell assigned to its nearest contour vertex (k-d tree); signed distance
   (positive = T-zone side, negative = B-zone side)
6. `tb_analysis_keep`: |distance| ≤ 150 µm and nearest segment QC-passed

The parameter scan repeats the inference for 5 settings (pixel size, σ, edge distance).

In [ ]:
def process_lymph_node(
    ad,
    pixel_size=20,
    sigma=2,
    border_zone_um=150,
    plot=True,

    # segment-based QC
    use_border_segment_qc=True,
    border_segment_length_um=100,
    min_segment_points=3,

    # NEW: remove neighboring segments around failed segments
    failed_segment_neighbor_buffer=1,

    # segment QC criteria
    hole_distance_um=75,
    curvature_step=5,
    max_curvature_quantile=0.90,
    hairpin_radius_um=100,
    hairpin_min_arc_gap_um=200,

    # two-sided support QC
    require_two_sided_support=True,
    min_cells_per_side=5,

    # optional cell support QC
    border_density_radius_um=100,
    min_cells_near_border=None,

    # contour filtering
    min_contour_length=100,
):
    """
    Infer T-B border and keep only analyzable local border segments.

    Analysis inclusion requires:
    - cell is within ±border_zone_um of the T-B border
    - nearest 100 µm border segment passes geometry QC
    - neighboring segments around failed segments are also removed
    - nearest segment has cells on both T-side and B-side

    For downstream Tg localization, use:
        tb_analysis_keep
    """

    import numpy as np
    import pandas as pd
    from scipy.ndimage import gaussian_filter, distance_transform_edt
    from scipy.spatial import cKDTree
    from skimage.measure import find_contours

    # =========================
    # Helper functions
    # =========================

    def mask_aware_gaussian(count_grid, valid_mask, sigma):
        valid_float = valid_mask.astype(float)
        numerator = gaussian_filter(count_grid * valid_float, sigma=sigma)
        denominator = gaussian_filter(valid_float, sigma=sigma)
        return numerator / (denominator + 1e-8)

    def contour_is_closed(contour, tolerance_um=None):
        if tolerance_um is None:
            tolerance_um = pixel_size * 2
        return np.linalg.norm(contour[0] - contour[-1]) <= tolerance_um

    def compute_arc_length(contour):
        steps = np.linalg.norm(np.diff(contour, axis=0), axis=1)
        return np.r_[0, np.cumsum(steps)]

    def compute_curvature(contour, is_closed):
        n = len(contour)
        step = curvature_step
        curvature_score = np.zeros(n)

        if n <= 2 * step + 1:
            return curvature_score

        if is_closed:
            prev_pts = np.roll(contour, step, axis=0)
            next_pts = np.roll(contour, -step, axis=0)
            valid_idx = np.ones(n, dtype=bool)
        else:
            prev_pts = np.full_like(contour, np.nan)
            next_pts = np.full_like(contour, np.nan)
            prev_pts[step:] = contour[:-step]
            next_pts[:-step] = contour[step:]

            idx = np.arange(n)
            valid_idx = (idx >= step) & (idx < n - step)

        v1 = contour - prev_pts
        v2 = next_pts - contour

        norm1 = np.linalg.norm(v1, axis=1)
        norm2 = np.linalg.norm(v2, axis=1)

        valid = valid_idx & (norm1 > 0) & (norm2 > 0)

        cosang = np.ones(n)
        cosang[valid] = np.sum(v1[valid] * v2[valid], axis=1) / (
            norm1[valid] * norm2[valid]
        )
        cosang = np.clip(cosang, -1, 1)

        curvature_score[valid] = np.arccos(cosang[valid])

        if not is_closed:
            curvature_score[:step] = np.nan
            curvature_score[-step:] = np.nan

        return curvature_score

    def compute_hairpin_points(contour, arc, is_closed):
        """
        Detect hairpins/self-approaching border regions.

        A point is marked bad if another contour point is spatially close
        but far away along arc length.
        """

        n = len(contour)
        hairpin_bad = np.zeros(n, dtype=bool)

        if n < 5:
            return hairpin_bad

        tree = cKDTree(contour)
        total_arc = arc[-1] if len(arc) > 0 else 0

        for idx, p in enumerate(contour):
            nearby = tree.query_ball_point(p, hairpin_radius_um)

            for j in nearby:
                if j == idx:
                    continue

                arc_gap = abs(arc[j] - arc[idx])

                if is_closed:
                    arc_gap = min(arc_gap, total_arc - arc_gap)

                if arc_gap >= hairpin_min_arc_gap_um:
                    hairpin_bad[idx] = True
                    break

        return hairpin_bad

    def build_segment_qc(contour, contour_id, is_closed):
        """
        Split one contour into fixed arc-length segments and assign QC.

        New behavior:
        If a segment fails QC, neighboring segments within
        failed_segment_neighbor_buffer are also marked unreliable.
        """

        arc = compute_arc_length(contour)

        segment_id_per_point = np.floor(
            arc / border_segment_length_um
        ).astype(int)

        unique_segments = np.unique(segment_id_per_point)

        # Distance to T/B-valid tissue edge or hole
        border_i = np.clip(
            ((contour[:, 0] - x_min) / pixel_size).astype(int),
            0,
            w - 1
        )

        border_j = np.clip(
            ((contour[:, 1] - y_min) / pixel_size).astype(int),
            0,
            h - 1
        )

        dist_to_tissue_edge = tissue_dist_grid[border_j, border_i]

        # Curvature
        curvature_score = compute_curvature(contour, is_closed)
        valid_curvature = curvature_score[~np.isnan(curvature_score)]

        if len(valid_curvature) > 0:
            curvature_cutoff = np.nanquantile(
                valid_curvature,
                max_curvature_quantile
            )
        else:
            curvature_cutoff = np.inf

        # Hairpins
        hairpin_bad = compute_hairpin_points(contour, arc, is_closed)

        # Optional local density support
        if min_cells_near_border is not None:
            cell_tree = cKDTree(cell_df[["x", "y"]].to_numpy())

            n_cells_near_border = np.array([
                len(cell_tree.query_ball_point(p, border_density_radius_um))
                for p in contour
            ])
        else:
            n_cells_near_border = np.repeat(np.nan, len(contour))

        segment_records = []
        segment_reliable_geometry_dict = {}

        # --------------------------------------------------------
        # First pass: raw geometry QC per segment
        # --------------------------------------------------------

        for sid in unique_segments:
            m = segment_id_per_point == sid
            n_points = int(m.sum())

            if n_points < min_segment_points:
                touches_edge = True
                has_hole = True
                excessive_curve = True
                has_hairpin = True
                enough_cells = False
                segment_reliable_geometry = False
            else:
                min_edge_dist = float(np.nanmin(dist_to_tissue_edge[m]))
                max_curve = float(np.nanmax(curvature_score[m]))
                has_hairpin = bool(np.any(hairpin_bad[m]))

                touches_edge = min_edge_dist <= pixel_size
                has_hole = min_edge_dist < hole_distance_um
                excessive_curve = max_curve > curvature_cutoff

                if min_cells_near_border is not None:
                    enough_cells = (
                        np.nanmean(n_cells_near_border[m])
                        >= min_cells_near_border
                    )
                else:
                    enough_cells = True

                segment_reliable_geometry = not (
                    touches_edge
                    or has_hole
                    or excessive_curve
                    or has_hairpin
                    or not enough_cells
                )

            segment_reliable_geometry_dict[sid] = segment_reliable_geometry

            segment_records.append({
                "contour_id": contour_id,
                "segment_id": int(sid),
                "n_points": n_points,
                "segment_reliable_geometry_raw": segment_reliable_geometry,
                "touches_tissue_mask_edge": touches_edge,
                "has_hole_or_near_edge": has_hole,
                "excessive_curvature": excessive_curve,
                "has_hairpin": has_hairpin,
                "enough_cells_near_border": enough_cells,
                "min_dist_to_tissue_edge_um": (
                    float(np.nanmin(dist_to_tissue_edge[m]))
                    if n_points > 0 else np.nan
                ),
                "max_curvature": (
                    float(np.nanmax(curvature_score[m]))
                    if n_points > 0 else np.nan
                ),
                "curvature_cutoff": curvature_cutoff,
                "mean_cells_near_border": (
                    float(np.nanmean(n_cells_near_border[m]))
                    if n_points > 0 else np.nan
                ),
            })

        # --------------------------------------------------------
        # Second pass: expand failed segments to neighboring segments
        # --------------------------------------------------------

        failed_segments = {
            sid
            for sid, ok in segment_reliable_geometry_dict.items()
            if not ok
        }

        expanded_failed_segments = set(failed_segments)

        if failed_segment_neighbor_buffer > 0:
            for sid in failed_segments:
                for d in range(1, failed_segment_neighbor_buffer + 1):
                    expanded_failed_segments.add(sid - d)
                    expanded_failed_segments.add(sid + d)

        # Only keep segment IDs that actually exist in this contour
        existing_segments = set(unique_segments.tolist())
        expanded_failed_segments = expanded_failed_segments.intersection(
            existing_segments
        )

        segment_reliable_geometry_buffered_dict = {
            sid: sid not in expanded_failed_segments
            for sid in unique_segments
        }

        segment_reliable_per_point = np.array([
            segment_reliable_geometry_buffered_dict[sid]
            for sid in segment_id_per_point
        ])

        segment_table = pd.DataFrame(segment_records)

        if not segment_table.empty:
            segment_table["segment_failed_after_neighbor_buffer"] = (
                segment_table["segment_id"].isin(expanded_failed_segments)
            )

            segment_table["segment_reliable_geometry"] = (
                ~segment_table["segment_failed_after_neighbor_buffer"]
            )

            segment_table["failed_segment_neighbor_buffer"] = (
                failed_segment_neighbor_buffer
            )

        return {
            "segment_id_per_point": segment_id_per_point,
            "segment_reliable_per_point": segment_reliable_per_point,
            "segment_table": segment_table,
        }

    # =========================
    # 1. Build dataframe
    # =========================

    ln = ad.obs["timepoint"].unique()[0]
    xy = ad.obsm["spatial"]

    cell_df = pd.DataFrame({
        "label": ad.obs_names,
        "x": xy[:, 0],
        "y": xy[:, 1],
        "fine_type": ad.obs["Tg_annotation_full"].values,
    })

    cell_df["LN"] = ln

    cell_df["cell_type"] = (
        cell_df["fine_type"]
        .map({
            "T_cells": "T",
            "B_cells": "B",
        })
        .fillna("none")
    )

    # =========================
    # 2. Initialize outputs
    # =========================

    cell_df["dist_to_border"] = np.nan
    cell_df["tb_border_zone"] = False

    cell_df["tb_analysis_keep"] = False
    cell_df["tb_analysis_side"] = None

    cell_df["tb_region_confident"] = "excluded"
    cell_df["tb_keep_high_confidence"] = False

    cell_df["tb_exclude"] = True
    cell_df["tb_exclude_reason"] = "not processed"

    cell_df["tb_field_strength"] = np.nan
    cell_df["dist_to_tissue_edge"] = np.nan

    cell_df["nearest_border_reliable"] = False
    cell_df["nearest_border_reliable_geometry"] = False
    cell_df["nearest_contour_id"] = np.nan
    cell_df["nearest_segment_id"] = np.nan

    cell_df["segment_has_two_sided_support"] = False
    cell_df["segment_n_T_side"] = 0
    cell_df["segment_n_B_side"] = 0

    # =========================
    # 3. Grid setup
    # =========================

    x_min, x_max = cell_df.x.min(), cell_df.x.max()
    y_min, y_max = cell_df.y.min(), cell_df.y.max()

    w = int(np.ceil((x_max - x_min) / pixel_size)) + 1
    h = int(np.ceil((y_max - y_min) / pixel_size)) + 1

    cell_df["i"] = np.clip(
        ((cell_df.x - x_min) / pixel_size).astype(int),
        0,
        w - 1
    )

    cell_df["j"] = np.clip(
        ((cell_df.y - y_min) / pixel_size).astype(int),
        0,
        h - 1
    )

    # =========================
    # 4. Build T/B grids
    # =========================

    T_grid = np.zeros((h, w), dtype=float)
    B_grid = np.zeros((h, w), dtype=float)

    for _, row in cell_df.iterrows():
        if row["cell_type"] == "T":
            T_grid[int(row["j"]), int(row["i"])] += 1
        elif row["cell_type"] == "B":
            B_grid[int(row["j"]), int(row["i"])] += 1

    tb_occupied = (T_grid + B_grid) > 0

    if tb_occupied.sum() == 0:
        print(f"WARNING: no T/B cells found in {ln}")
        cell_df["tb_exclude_reason"] = "no T/B cells"
        return cell_df

    # =========================
    # 5. T/B-valid tissue mask
    # =========================

    tb_density_for_mask = gaussian_filter(
        tb_occupied.astype(float),
        sigma=2
    )

    tissue_binary = tb_density_for_mask > 0.2

    tissue_dist_grid = (
        distance_transform_edt(tissue_binary)
        * pixel_size
    )

    # =========================
    # 6. Smoothed T/B score
    # =========================

    T_smooth = mask_aware_gaussian(
        T_grid,
        tissue_binary,
        sigma=sigma
    )

    B_smooth = mask_aware_gaussian(
        B_grid,
        tissue_binary,
        sigma=sigma
    )

    tb_score = (
        (T_smooth - B_smooth)
        /
        (T_smooth + B_smooth + 1e-8)
    )

    tb_score[~tissue_binary] = np.nan

    if np.nanmin(tb_score) > 0 or np.nanmax(tb_score) < 0:
        print(f"WARNING: no T-B interface found in {ln}")
        cell_df["tb_exclude_reason"] = "no T-B interface"
        return cell_df

    # =========================
    # 7. Extract T-B contours
    # =========================

    raw_contours = find_contours(
        tb_score,
        level=0
    )

    smooth_contours = []

    for c in raw_contours:
        if len(c) < min_contour_length:
            continue

        x_real = c[:, 1] * pixel_size + x_min
        y_real = c[:, 0] * pixel_size + y_min

        smooth_contours.append(
            np.c_[x_real, y_real]
        )

    smooth_contours = sorted(
        smooth_contours,
        key=len,
        reverse=True
    )

    if len(smooth_contours) == 0:
        print(f"WARNING: no usable T-B contour in {ln}")
        cell_df["tb_exclude_reason"] = "no usable contour"
        return cell_df

    # =========================
    # 8. Segment-level geometry QC
    # =========================

    contour_reliability_geometry = []
    contour_segment_ids = []
    segment_tables = []

    for cid, contour in enumerate(smooth_contours):
        is_closed = contour_is_closed(contour)

        if use_border_segment_qc:
            seg_qc = build_segment_qc(
                contour,
                contour_id=cid,
                is_closed=is_closed
            )

            reliable_geometry = seg_qc["segment_reliable_per_point"]
            segment_ids = seg_qc["segment_id_per_point"]

            if not seg_qc["segment_table"].empty:
                segment_tables.append(seg_qc["segment_table"])
        else:
            reliable_geometry = np.ones(len(contour), dtype=bool)
            segment_ids = np.zeros(len(contour), dtype=int)

        contour_reliability_geometry.append(reliable_geometry)
        contour_segment_ids.append(segment_ids)

    border_segment_qc_df = (
        pd.concat(segment_tables, ignore_index=True)
        if len(segment_tables) > 0
        else pd.DataFrame()
    )

    # =========================
    # 9. Assign cells to nearest border segment
    # =========================

    all_border_xy = []
    all_border_reliable_geometry = []
    all_border_contour_id = []
    all_border_segment_id = []

    for cid, contour in enumerate(smooth_contours):
        all_border_xy.append(contour)
        all_border_reliable_geometry.append(contour_reliability_geometry[cid])
        all_border_contour_id.extend([cid] * len(contour))
        all_border_segment_id.extend(list(contour_segment_ids[cid]))

    all_border_xy = np.vstack(all_border_xy)
    all_border_reliable_geometry = np.concatenate(all_border_reliable_geometry)

    all_border_contour_id = np.array(all_border_contour_id)
    all_border_segment_id = np.array(all_border_segment_id)

    border_tree = cKDTree(all_border_xy)
    cell_xy = cell_df[["x", "y"]].to_numpy()

    dist_um, nearest_idx = border_tree.query(cell_xy)

    cell_df["nearest_contour_id"] = all_border_contour_id[nearest_idx]
    cell_df["nearest_segment_id"] = all_border_segment_id[nearest_idx]
    cell_df["nearest_border_reliable_geometry"] = (
        all_border_reliable_geometry[nearest_idx]
    )

    # =========================
    # 10. Signed distance
    # =========================

    signs = []

    for i, j in zip(cell_df["i"], cell_df["j"]):
        val = tb_score[j, i]

        if np.isnan(val):
            signs.append(np.nan)
        elif val > 0:
            signs.append(1)
        elif val < 0:
            signs.append(-1)
        else:
            signs.append(0)

    cell_df["dist_to_border"] = (
        dist_um * np.array(signs)
    )

    cell_df["tb_border_zone"] = (
        cell_df["dist_to_border"].abs()
        <= border_zone_um
    )

    cell_df["tb_analysis_side"] = np.select(
        [
            cell_df["dist_to_border"] > 0,
            cell_df["dist_to_border"] < 0,
        ],
        [
            "T-zone side",
            "B-zone side",
        ],
        default=None
    )

    # =========================
    # 11. Field strength and edge distance
    # =========================

    for idx, (i, j) in enumerate(zip(cell_df["i"], cell_df["j"])):
        cell_df.loc[cell_df.index[idx], "tb_field_strength"] = abs(tb_score[j, i])
        cell_df.loc[cell_df.index[idx], "dist_to_tissue_edge"] = tissue_dist_grid[j, i]

    # =========================
    # 12. Two-sided segment support QC
    # =========================

    if require_two_sided_support:
        support_source = cell_df[
            cell_df["tb_border_zone"]
            &
            cell_df["tb_analysis_side"].isin(
                ["T-zone side", "B-zone side"]
            )
        ].copy()

        side_counts = (
            support_source
            .groupby([
                "nearest_contour_id",
                "nearest_segment_id",
                "tb_analysis_side"
            ])
            .size()
            .unstack(fill_value=0)
            .reset_index()
        )

        if "T-zone side" not in side_counts.columns:
            side_counts["T-zone side"] = 0

        if "B-zone side" not in side_counts.columns:
            side_counts["B-zone side"] = 0

        side_counts["segment_n_T_side"] = side_counts["T-zone side"]
        side_counts["segment_n_B_side"] = side_counts["B-zone side"]

        side_counts["segment_has_two_sided_support"] = (
            (side_counts["segment_n_T_side"] >= min_cells_per_side)
            &
            (side_counts["segment_n_B_side"] >= min_cells_per_side)
        )

        side_counts = side_counts[
            [
                "nearest_contour_id",
                "nearest_segment_id",
                "segment_n_T_side",
                "segment_n_B_side",
                "segment_has_two_sided_support",
            ]
        ]

        cell_df = cell_df.merge(
            side_counts,
            on=["nearest_contour_id", "nearest_segment_id"],
            how="left",
            suffixes=("", "_from_segment")
        )

        cell_df["segment_n_T_side"] = (
            cell_df["segment_n_T_side_from_segment"]
            .fillna(0)
            .astype(int)
        )

        cell_df["segment_n_B_side"] = (
            cell_df["segment_n_B_side_from_segment"]
            .fillna(0)
            .astype(int)
        )

        cell_df["segment_has_two_sided_support"] = (
            cell_df["segment_has_two_sided_support_from_segment"]
            .fillna(False)
            .astype(bool)
        )

        cell_df = cell_df.drop(
            columns=[
                "segment_n_T_side_from_segment",
                "segment_n_B_side_from_segment",
                "segment_has_two_sided_support_from_segment",
            ],
            errors="ignore"
        )

    else:
        cell_df["segment_has_two_sided_support"] = True

    # =========================
    # 13. Final analysis mask
    # =========================

    cell_df["nearest_border_reliable"] = (
        cell_df["nearest_border_reliable_geometry"]
        &
        cell_df["segment_has_two_sided_support"]
    )

    cell_df["tb_analysis_keep"] = (
        cell_df["tb_border_zone"]
        &
        cell_df["nearest_border_reliable"]
        &
        (~cell_df["dist_to_border"].isna())
    )

    # =========================
    # 14. Visual annotation
    # =========================

    cell_df["tb_region_confident"] = "excluded"

    cell_df.loc[
        cell_df["tb_analysis_keep"]
        &
        (cell_df["dist_to_border"] > 0),
        "tb_region_confident"
    ] = "T_side_border_zone"

    cell_df.loc[
        cell_df["tb_analysis_keep"]
        &
        (cell_df["dist_to_border"] < 0),
        "tb_region_confident"
    ] = "B_side_border_zone"

    cell_df["tb_keep_high_confidence"] = cell_df["tb_analysis_keep"]

    cell_df["tb_exclude"] = ~cell_df["tb_analysis_keep"]
    cell_df["tb_exclude_reason"] = "kept"

    cell_df.loc[
        ~cell_df["tb_border_zone"],
        "tb_exclude_reason"
    ] = "outside border zone"

    cell_df.loc[
        cell_df["tb_border_zone"]
        &
        (~cell_df["nearest_border_reliable_geometry"]),
        "tb_exclude_reason"
    ] = "failed border segment geometry"

    cell_df.loc[
        cell_df["tb_border_zone"]
        &
        cell_df["nearest_border_reliable_geometry"]
        &
        (~cell_df["segment_has_two_sided_support"]),
        "tb_exclude_reason"
    ] = "failed two-sided segment support"

    cell_df.loc[
        cell_df["dist_to_border"].isna(),
        "tb_exclude_reason"
    ] = "no border distance"

    # =========================
    # 15. Add two-sided support to segment QC table
    # =========================

    if (
        not border_segment_qc_df.empty
        and
        "side_counts" in locals()
    ):
        border_segment_qc_df = border_segment_qc_df.merge(
            side_counts.rename(
                columns={
                    "nearest_contour_id": "contour_id",
                    "nearest_segment_id": "segment_id",
                }
            ),
            on=["contour_id", "segment_id"],
            how="left"
        )

        border_segment_qc_df["segment_n_T_side"] = (
            border_segment_qc_df["segment_n_T_side"]
            .fillna(0)
            .astype(int)
        )

        border_segment_qc_df["segment_n_B_side"] = (
            border_segment_qc_df["segment_n_B_side"]
            .fillna(0)
            .astype(int)
        )

        border_segment_qc_df["segment_has_two_sided_support"] = (
            border_segment_qc_df["segment_has_two_sided_support"]
            .fillna(False)
            .astype(bool)
        )

        border_segment_qc_df["segment_reliable_final"] = (
            border_segment_qc_df["segment_reliable_geometry"]
            &
            border_segment_qc_df["segment_has_two_sided_support"]
        )

    # =========================
    # 16. Store outputs for plotting
    # =========================

    final_border_reliability = []

    for cid, contour in enumerate(smooth_contours):
        seg_ids = contour_segment_ids[cid]
        geom_rel = contour_reliability_geometry[cid]

        tmp = pd.DataFrame({
            "contour_id": cid,
            "segment_id": seg_ids,
            "geometry_reliable": geom_rel,
        })

        if not border_segment_qc_df.empty:
            tmp = tmp.merge(
                border_segment_qc_df[
                    [
                        "contour_id",
                        "segment_id",
                        "segment_has_two_sided_support",
                    ]
                ].drop_duplicates(),
                on=["contour_id", "segment_id"],
                how="left"
            )

            support = (
                tmp["segment_has_two_sided_support"]
                .fillna(False)
                .astype(bool)
                .values
            )
        else:
            support = np.ones(len(tmp), dtype=bool)

        final_border_reliability.append(
            geom_rel & support
        )

    cell_df.attrs["tb_border_contours"] = smooth_contours
    cell_df.attrs["tb_border_reliability"] = final_border_reliability
    cell_df.attrs["tb_border_segment_ids"] = contour_segment_ids
    cell_df.attrs["tb_border_segment_qc"] = border_segment_qc_df

    return cell_df

In [ ]:
# ============================================================
# TB-border inference for run1 + run2
# ============================================================


# ------------------------------------------------------------
# 1. Read / define input objects
# ------------------------------------------------------------

ad_run1 = ad_run1_tb
ad_run2 = ad_run2_tb

runs = {
    "run1": ad_run1,
    "run2": ad_run2,
}


# ------------------------------------------------------------
# 2. Parameter settings
# ------------------------------------------------------------

parameter_grid = [
    {"pixel_size": 15, "sigma": 1.5, "hole_distance_um": 50},
    {"pixel_size": 15, "sigma": 2.0, "hole_distance_um": 75},
    {"pixel_size": 20, "sigma": 1.5, "hole_distance_um": 75},
    {"pixel_size": 20, "sigma": 2.0, "hole_distance_um": 75},
    {"pixel_size": 30, "sigma": 1.5, "hole_distance_um": 100},
]

main_params = {
    "pixel_size": 20,
    "sigma": 1.5,
    "hole_distance_um": 75,
}


# ------------------------------------------------------------
# 3. Helper: run TB-border inference for one AnnData object
# ------------------------------------------------------------

def run_tb_border_inference(
    adata,
    run_label,
    parameter_grid,
    main_params,
    timepoint_col="timepoint",
    border_zone_um=150,
):
    """
    Run TB-border inference for one Xenium run.

    Returns
    -------
    main_cells_df : pd.DataFrame
        Cell-level results for the main parameter setting only.

    scan_cells_df : pd.DataFrame
        Cell-level results for all parameter settings.

    scan_summary_df : pd.DataFrame
        Per-LN / per-parameter summary table.

    border_points_df : pd.DataFrame
        Border contour point-level table for plotting / QC.
    """

    main_cell_results = []
    all_scan_cell_results = []
    scan_summary = []
    border_point_results = []

    lns = adata.obs[timepoint_col].dropna().unique()

    for ln in lns:

        print("\n==============================")
        print(f"Run: {run_label} | Processing: {ln}")
        print("==============================")

        ad_ln = adata[
            adata.obs[timepoint_col] == ln
        ].copy()

        for params in parameter_grid:

            is_main_run = params == main_params

            print(
                f"\nScan: {run_label} | {ln} | "
                f"pixel_size={params['pixel_size']} | "
                f"sigma={params['sigma']} | "
                f"hole_distance_um={params['hole_distance_um']} | "
                f"is_main_run={is_main_run}"
            )

            cell_df = process_lymph_node(
                ad_ln,
                border_zone_um=border_zone_um,
                pixel_size=params["pixel_size"],
                sigma=params["sigma"],
                plot=False,

                # segment-based QC
                use_border_segment_qc=True,
                border_segment_length_um=100,
                min_segment_points=3,
                failed_segment_neighbor_buffer=1,

                # hole / edge filtering
                hole_distance_um=params["hole_distance_um"],

                # curvature filtering
                curvature_step=5,
                max_curvature_quantile=0.90,

                # hairpin filtering
                hairpin_radius_um=100,
                hairpin_min_arc_gap_um=200,

                # require both sides of the segment
                require_two_sided_support=True,
                min_cells_per_side=5,

                # optional local cell-density filter
                min_cells_near_border=None,

                # contour filtering
                min_contour_length=100,
            )

            if cell_df is None:
                scan_summary.append({
                    "run": run_label,
                    "LN": ln,
                    **params,
                    "is_main_run": is_main_run,
                    "status": "failed",
                })
                continue

            # ------------------------------------------------
            # Add metadata
            # ------------------------------------------------

            cell_df["run"] = run_label
            cell_df["LN"] = ln
            cell_df["pixel_size"] = params["pixel_size"]
            cell_df["sigma"] = params["sigma"]
            cell_df["hole_distance_um"] = params["hole_distance_um"]
            cell_df["is_main_run"] = is_main_run

            # unique segment ID across run / LN / contour / segment
            cell_df["nearest_border_segment"] = (
                cell_df["run"].astype(str)
                + "__"
                + cell_df["LN"].astype(str)
                + "__c"
                + cell_df["nearest_contour_id"].astype(str)
                + "__s"
                + cell_df["nearest_segment_id"].astype(str)
            )

            # ------------------------------------------------
            # Extract border contours from attrs before clearing
            # ------------------------------------------------

            tb_border_contours = cell_df.attrs.get("tb_border_contours", [])
            tb_border_reliability = cell_df.attrs.get("tb_border_reliability", [])
            tb_border_segment_ids = cell_df.attrs.get("tb_border_segment_ids", [])

            for cid, contour in enumerate(tb_border_contours):

                if cid >= len(tb_border_reliability):
                    continue

                reliable = tb_border_reliability[cid]

                if cid < len(tb_border_segment_ids):
                    segment_ids = tb_border_segment_ids[cid]
                else:
                    segment_ids = [None] * len(contour)

                for point_id, xy in enumerate(contour):

                    if point_id >= len(reliable):
                        continue

                    segment_id = (
                        segment_ids[point_id]
                        if point_id < len(segment_ids)
                        else None
                    )

                    border_point_results.append({
                        "run": run_label,
                        "LN": ln,
                        "pixel_size": params["pixel_size"],
                        "sigma": params["sigma"],
                        "hole_distance_um": params["hole_distance_um"],
                        "is_main_run": is_main_run,
                        "contour_id": cid,
                        "segment_id": segment_id,
                        "point_id": point_id,
                        "x": xy[0],
                        "y": xy[1],
                        "reliable": reliable[point_id],
                        "border_segment": (
                            f"{run_label}__{ln}__c{cid}__s{segment_id}"
                        ),
                    })

            # Avoid pd.concat problems from dataframe attrs
            cell_df.attrs = {}

            all_scan_cell_results.append(cell_df)

            if is_main_run:
                main_cell_results.append(cell_df)

            # ------------------------------------------------
            # Summary
            # ------------------------------------------------

            n_total = len(cell_df)
            n_analysis = cell_df["tb_analysis_keep"].sum()

            scan_summary.append({
                "run": run_label,
                "LN": ln,
                **params,
                "is_main_run": is_main_run,
                "status": "ok",

                "n_cells_total": n_total,
                "n_analysis_cells": n_analysis,
                "frac_analysis_cells": n_analysis / n_total if n_total > 0 else 0,

                "n_T_side_border_zone": (
                    cell_df["tb_region_confident"] == "T_side_border_zone"
                ).sum(),

                "n_B_side_border_zone": (
                    cell_df["tb_region_confident"] == "B_side_border_zone"
                ).sum(),

                "frac_failed_geometry": (
                    cell_df["tb_exclude_reason"]
                    == "failed border segment geometry"
                ).mean(),

                "frac_failed_two_sided_support": (
                    cell_df["tb_exclude_reason"]
                    == "failed two-sided segment support"
                ).mean(),

                "frac_outside_border_zone": (
                    cell_df["tb_exclude_reason"]
                    == "outside border zone"
                ).mean(),

                "median_abs_dist_to_border": (
                    cell_df["dist_to_border"].abs().median()
                ),
            })

    main_cells_df = (
        pd.concat(main_cell_results, ignore_index=True)
        if len(main_cell_results) > 0
        else pd.DataFrame()
    )

    scan_cells_df = (
        pd.concat(all_scan_cell_results, ignore_index=True)
        if len(all_scan_cell_results) > 0
        else pd.DataFrame()
    )

    scan_summary_df = pd.DataFrame(scan_summary)
    border_points_df = pd.DataFrame(border_point_results)

    return {
        "main_cells": main_cells_df,
        "scan_cells": scan_cells_df,
        "scan_summary": scan_summary_df,
        "border_points": border_points_df,
    }


# ------------------------------------------------------------
# 4. Run inference for both runs
# ------------------------------------------------------------

tb_results = {}

for run_label, adata in runs.items():
    tb_results[run_label] = run_tb_border_inference(
        adata=adata,
        run_label=run_label,
        parameter_grid=parameter_grid,
        main_params=main_params,
        timepoint_col="timepoint",
        border_zone_um=150,
    )


# ------------------------------------------------------------
# 5. Run-specific result tables
# ------------------------------------------------------------

all_results_df_run1 = tb_results["run1"]["main_cells"]
all_scan_results_df_run1 = tb_results["run1"]["scan_cells"]
scan_results_df_run1 = tb_results["run1"]["scan_summary"]
all_border_results_df_run1 = tb_results["run1"]["border_points"]

all_results_df_run2 = tb_results["run2"]["main_cells"]
all_scan_results_df_run2 = tb_results["run2"]["scan_cells"]
scan_results_df_run2 = tb_results["run2"]["scan_summary"]
all_border_results_df_run2 = tb_results["run2"]["border_points"]


# ------------------------------------------------------------
# 6. Combine run1 + run2
# ------------------------------------------------------------

all_results_df = pd.concat(
    [all_results_df_run1, all_results_df_run2],
    ignore_index=True
)

all_scan_results_df = pd.concat(
    [all_scan_results_df_run1, all_scan_results_df_run2],
    ignore_index=True
)

scan_results_df = pd.concat(
    [scan_results_df_run1, scan_results_df_run2],
    ignore_index=True
)

all_border_results_df = pd.concat(
    [all_border_results_df_run1, all_border_results_df_run2],
    ignore_index=True
)


# ------------------------------------------------------------
# 7. Quick sanity checks
# ------------------------------------------------------------

print("Main cell results:")
display(
    all_results_df
    .groupby(["run", "LN"], observed=False)
    .size()
    .reset_index(name="n_cells")
)

print("Analysis cells:")
display(
    all_results_df
    .groupby(["run", "LN"], observed=False)
    ["tb_analysis_keep"]
    .sum()
    .reset_index(name="n_tb_analysis_cells")
)

print("Scan summary:")
display(scan_results_df.head())

### Parameter scan

In [ ]:
def plot_parameter_scan_grid_with_border_compact_per_run(
    all_scan_results_df,
    all_border_results_df,
    point_size=0.7,
    panel_width=2.4,
    save_path_prefix=None,
    pad_frac=0.03,
    run_order=None
):
    import matplotlib.pyplot as plt

    cells_all = all_scan_results_df.copy()
    borders_all = all_border_results_df.copy()

    if "run" not in cells_all.columns:
        raise ValueError("all_scan_results_df needs a 'run' column")
    if "run" not in borders_all.columns:
        raise ValueError("all_border_results_df needs a 'run' column")

    cells_all["param_label"] = (
        "px=" + cells_all["pixel_size"].astype(str)
        + "\nσ=" + cells_all["sigma"].astype(str)
        + "\nhole=" + cells_all["hole_distance_um"].astype(str)
    )

    borders_all["param_label"] = (
        "px=" + borders_all["pixel_size"].astype(str)
        + "\nσ=" + borders_all["sigma"].astype(str)
        + "\nhole=" + borders_all["hole_distance_um"].astype(str)
    )

    if run_order is None:
        run_order = sorted(cells_all["run"].dropna().unique())

    region_order = [
        "excluded",
        "T_side_border_zone",
        "B_side_border_zone",
    ]

    region_colors = {
        "excluded": "lightgrey",
        "T_side_border_zone": "#3fd6c1",   # T cells: cyan
        "B_side_border_zone": "#4a90e2",   # B cells: blue
    }

    region_alpha = {
        "excluded": 0.18,
        "T_side_border_zone": 0.75,
        "B_side_border_zone": 0.75,
    }

    region_size = {
        "excluded": point_size,
        "T_side_border_zone": point_size * 1.3,
        "B_side_border_zone": point_size * 1.3,
    }

    figures = {}

    for run in run_order:

        cells = cells_all[cells_all["run"] == run].copy()
        borders = borders_all[borders_all["run"] == run].copy()

        if cells.empty:
            print(f"Skipping {run}: no cell data")
            continue

        lns = sorted(cells["LN"].dropna().unique())

        params = (
            cells[["pixel_size", "sigma", "hole_distance_um", "param_label"]]
            .drop_duplicates()
            .sort_values(["pixel_size", "sigma", "hole_distance_um"])
        )

        param_labels = params["param_label"].tolist()

        ln_bounds = {}
        height_ratios = []

        for ln in lns:
            sub = cells[cells["LN"] == ln]

            x_min, x_max = sub["x"].min(), sub["x"].max()
            y_min, y_max = sub["y"].min(), sub["y"].max()

            x_range = x_max - x_min
            y_range = y_max - y_min

            x_pad = x_range * pad_frac
            y_pad = y_range * pad_frac

            ln_bounds[ln] = (
                x_min - x_pad,
                x_max + x_pad,
                y_min - y_pad,
                y_max + y_pad,
            )

            ratio = (y_range + 2 * y_pad) / (x_range + 2 * x_pad)
            height_ratios.append(max(ratio, 0.4))

        n_rows = len(lns)
        n_cols = len(param_labels)

        fig_width = panel_width * n_cols
        fig_height = panel_width * sum(height_ratios)

        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(fig_width, fig_height),
            squeeze=False,
            gridspec_kw={"height_ratios": height_ratios}
        )

        for r, ln in enumerate(lns):
            x_min, x_max, y_min, y_max = ln_bounds[ln]

            for c, param_label in enumerate(param_labels):

                ax = axes[r, c]

                sub_cells = cells[
                    (cells["LN"] == ln) &
                    (cells["param_label"] == param_label)
                ]

                sub_border = borders[
                    (borders["LN"] == ln) &
                    (borders["param_label"] == param_label)
                ]

                for region in region_order:
                    m = sub_cells["tb_region_confident"] == region

                    ax.scatter(
                        sub_cells.loc[m, "x"],
                        sub_cells.loc[m, "y"],
                        s=region_size[region],
                        c=region_colors[region],
                        alpha=region_alpha[region],
                        linewidths=0
                    )

                if not sub_border.empty:
                    for cid in sorted(sub_border["contour_id"].unique()):
                        b = (
                            sub_border[sub_border["contour_id"] == cid]
                            .sort_values("point_id")
                        )

                        ax.plot(
                            b["x"],
                            b["y"],
                            color="#9a9a9a",
                            linewidth=0.6,
                            linestyle=(0, (3, 2)),
                            alpha=0.6
                        )

                        br = b[b["reliable"]]

                        ax.plot(
                            br["x"],
                            br["y"],
                            color="black",
                            linewidth=1.0,
                            alpha=0.95
                        )

                ax.set_xlim(x_min, x_max)
                ax.set_ylim(y_max, y_min)
                ax.set_aspect("equal")
                ax.axis("off")

                if r == 0:
                    ax.set_title(param_label, fontsize=7, pad=1)

                if c == 0:
                    ax.text(
                        -0.025,
                        0.5,
                        str(ln),
                        transform=ax.transAxes,
                        rotation=90,
                        va="center",
                        ha="right",
                        fontsize=8,
                        fontweight="bold"
                    )

        handles = [
            plt.Line2D(
                [0], [0],
                marker="o",
                color="w",
                markerfacecolor=region_colors[region],
                markersize=5,
                label=region
            )
            for region in region_order
        ]

        border_handles = [
            plt.Line2D(
                [0], [0],
                color="black",
                linewidth=1.5,
                label="QC-passed border segment"
            ),
            plt.Line2D(
                [0], [0],
                color="#9a9a9a",
                linewidth=1,
                linestyle=(0, (3, 2)),
                label="full inferred border"
            )
        ]

        fig.suptitle(f"{run}", fontsize=12, y=0.995)

        fig.legend(
            handles=handles + border_handles,
            loc="lower center",
            ncol=5,
            frameon=False,
            fontsize=7,
            handlelength=1,
            columnspacing=0.6
        )

        plt.subplots_adjust(
            left=0.035,
            right=0.995,
            top=0.94,
            bottom=0.055,
            wspace=0.005,
            hspace=0.03
        )

        if save_path_prefix is not None:
            plt.savefig(
                f"{save_path_prefix}_{run}.png",
                dpi=300,
                bbox_inches="tight"
            )

        plt.show()

        figures[run] = (fig, axes)

    return figures

In [ ]:
plot_parameter_scan_grid_with_border_compact_per_run(
    all_scan_results_df=all_scan_results_df,
    all_border_results_df=all_border_results_df,
    point_size=0.7,
    panel_width=2.4
)

### Single section with inferred border → Figure 4b (bottom)

Segment-QC criteria are illustrated in **Supplementary Figure 4c** (`make_qc_segments.py`).

In [ ]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
run_sel = "run1"
ln_sel = "3:Day4_0391"          # LN name as in the "LN" column
params_sel = {"pixel_size": 20, "sigma": 1.5, "hole_distance_um": 75}

region_order = ["excluded", "T_side_border_zone", "B_side_border_zone"]
region_colors = {
    "excluded": "lightgrey",
    "T_side_border_zone": "#3fd6c1",   # T cells: cyan
    "B_side_border_zone": "#4a90e2",   # B cells: blue
}
region_alpha = {"excluded": 0.18, "T_side_border_zone": 0.75, "B_side_border_zone": 0.75}
point_size = 1.5
region_size = {"excluded": point_size, "T_side_border_zone": point_size * 1.3, "B_side_border_zone": point_size * 1.3}

# ------------------------------------------------------------
# Subset cells and border points
# ------------------------------------------------------------
def select(df):
    m = (df["run"] == run_sel) & (df["LN"] == ln_sel)
    for k, v in params_sel.items():
        m &= df[k] == v
    return df[m]

cells = select(all_scan_results_df)
border = select(all_border_results_df)
print(f"{len(cells)} cells, {border['contour_id'].nunique() if len(border) else 0} contours")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
x_range = cells["x"].max() - cells["x"].min()
y_range = cells["y"].max() - cells["y"].min()
fig_w = 5
fig, ax = plt.subplots(figsize=(fig_w, fig_w * y_range / x_range))

for region in region_order:
    m = cells["tb_region_confident"] == region
    ax.scatter(cells.loc[m, "x"], cells.loc[m, "y"], s=region_size[region],
               c=region_colors[region], alpha=region_alpha[region], linewidths=0)

for cid in sorted(border["contour_id"].unique()):
    b = border[border["contour_id"] == cid].sort_values("point_id")
    ax.plot(b["x"], b["y"], color="#9a9a9a", linewidth=0.6, linestyle=(0, (3, 2)), alpha=0.6)
    br = b[b["reliable"]]
    ax.plot(br["x"], br["y"], color="black", linewidth=1.0, alpha=0.95)

ax.set_aspect("equal")
ax.invert_yaxis()
ax.axis("off")

handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=region_colors[r], markersize=5, label=r)
           for r in region_order]
handles += [
    plt.Line2D([0], [0], color="black", linewidth=1.5, label="QC-passed border segment"),
    plt.Line2D([0], [0], color="#9a9a9a", linewidth=1, linestyle=(0, (3, 2)), label="full inferred border"),
]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0), ncol=3,
          frameon=False, fontsize=7, handlelength=1, columnspacing=0.8)

safe_ln = ln_sel.replace(":", "_")
fig.savefig(f"Fig4b_TB_border_{run_sel}_{safe_ln}_px20_s1.5_h75.png", bbox_inches="tight", dpi=600, transparent=True)
plt.show()

## 5. Tg cells relative to the T/B border

`tg_plot`: Tg cells within ±150 µm of a QC-passed border segment (`tb_analysis_keep`).

In [ ]:
def prepare_tg_distance_data(
    all_results,
    tg_types=("Tg_pre_Th1", "Tg_Th1", "Tg_Tfh", "Tg_pre_Tfh"),
    ordered_timepoints=(
        "10:Day0_0382",
        "16:Day0_0383",

        "2:Day2_0386",
        "15:Day2_0386",
        "8:Day2_0385",
        "9:Day2_0385",

        "4:Day3_0388",
        "6:Day3_0388",
        "7:Day3_0388",
        "13:Day3_0390",
        "14:Day3_0390",

        "3:Day4_0391",
        "5:Day4_0391",
        "12:Day4_0393",

        "1:Day6_0394",
        "11:Day6_0396",
    ),
    border_zone_um=150,
    plot_only_border_zone=True
):
    import numpy as np
    import pandas as pd

    if isinstance(all_results, list):
        all_cells = pd.concat(all_results, ignore_index=True)
    else:
        all_cells = all_results.copy()

    required = {"fine_type", "dist_to_border", "LN", "tb_analysis_keep"}
    missing = required - set(all_cells.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    tg_plot = all_cells[
        all_cells["fine_type"].isin(tg_types)
    ].copy()

    tg_plot = tg_plot.dropna(subset=["dist_to_border", "LN"])

    tg_plot["border_zone"] = np.where(
        tg_plot["dist_to_border"].abs() <= border_zone_um,
        "inside",
        "outside"
    )

    tg_plot = tg_plot[tg_plot["tb_analysis_keep"]].copy()

    if plot_only_border_zone:
        tg_plot = tg_plot[tg_plot["border_zone"] == "inside"].copy()

    tg_plot["LN"] = pd.Categorical(
        tg_plot["LN"],
        categories=list(ordered_timepoints),
        ordered=True
    )

    tg_plot = tg_plot.dropna(subset=["LN"])

    tg_plot["fine_type"] = pd.Categorical(
        tg_plot["fine_type"],
        categories=list(tg_types),
        ordered=True
    )

    tg_plot["day"] = (
        tg_plot["LN"]
        .astype(str)
        .str.extract(r"Day(\d+)")[0]
        .astype(float)
    )

    tg_plot["sample"] = tg_plot["LN"].astype(str)

    # Keep run1 and run2 separate if run column exists
    group_cols = (
        ["run", "sample", "day", "fine_type"]
        if "run" in tg_plot.columns
        else ["sample", "day", "fine_type"]
    )

    summary = (
        tg_plot
        .groupby(group_cols, observed=False)
        .agg(
            median_dist=("dist_to_border", "median"),
            mean_dist=("dist_to_border", "mean"),
            n_cells=("dist_to_border", "size")
        )
        .reset_index()
    )

    summary = summary[summary["n_cells"] > 0].copy()

    return tg_plot, summary

In [ ]:
# ============================================================
# Prepare Tg cells for ECDF / downstream spatial analysis
# ============================================================

tg_types = [
    "Tg_pre_Th1",
    "Tg_Th1",
    "Tg_pre_Tfh",
    "Tg_Tfh",
]

ordered_timepoints = [
    "10:Day0_0382",
    "16:Day0_0383",

    "2:Day2_0386",
    "15:Day2_0386",
    "8:Day2_0385",
    "9:Day2_0385",

    "4:Day3_0388",
    "7:Day3_0388",
    "13:Day3_0390",
    "14:Day3_0390",

    "3:Day4_0391",
    "5:Day4_0391",
    "12:Day4_0393",

    "1:Day6_0394",
    "11:Day6_0396",
]

tg_plot, tg_summary = prepare_tg_distance_data(
    all_results_df,
    tg_types=tg_types,
    ordered_timepoints=ordered_timepoints,
    border_zone_um=150,
    plot_only_border_zone=True,
)

# Keep run in the plotting table, if present
if "run" in all_results_df.columns and "run" not in tg_plot.columns:
    tg_plot = tg_plot.merge(
        all_results_df[["run", "LN", "dist_to_border", "fine_type"]].drop_duplicates(),
        on=["LN", "dist_to_border", "fine_type"],
        how="left"
    )

display(tg_summary.head())

### Tg subsets per day in the border corridor

In [ ]:
# ============================================================
# Stacked barplot: Tg cell populations per day
# Corridor data already contained in tg_plot
# Both Xenium runs combined, Day 0 removed
# ============================================================


module_palette = {
    "Tg_pre_Tfh": "#6baed6",
    "Tg_Tfh": "#08519c",
    "Tg_pre_Th1": "#fdae6b",
    "Tg_Th1": "#a63603"
}

tg_types = [
    "Tg_pre_Th1",
    "Tg_Th1",
    "Tg_pre_Tfh",
    "Tg_Tfh"
]

day_order = [2, 3, 4, 6]

# Copy corridor data
tg_plot_bar = tg_plot.copy()

# Extract day
tg_plot_bar["day"] = (
    tg_plot_bar["LN"]
    .astype(str)
    .str.extract(r"Day(\d+)", expand=False)
    .astype(int)
)

# Remove Day 0 and keep only Tg populations
tg_plot_bar = tg_plot_bar[
    tg_plot_bar["day"].isin(day_order)
    & tg_plot_bar["fine_type"].isin(tg_types)
].copy()

# Count cells per day and Tg subtype
tg_counts_per_day = (
    tg_plot_bar
    .groupby(["day", "fine_type"], observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
)

# Add missing combinations as zero
full_index = pd.MultiIndex.from_product(
    [day_order, tg_types],
    names=["day", "fine_type"]
)

tg_counts_per_day = (
    tg_counts_per_day
    .set_index(["day", "fine_type"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# Convert to wide format for stacked plotting
plot_df = (
    tg_counts_per_day
    .pivot(
        index="day",
        columns="fine_type",
        values="n_cells"
    )
    .reindex(index=day_order, columns=tg_types, fill_value=0)
)

display(plot_df)

# Colors in the same order as the stacked categories
plot_colors = [
    module_palette[celltype]
    for celltype in plot_df.columns
]

# Plot
fig, ax = plt.subplots(figsize=(3.6, 3))

plot_df.plot(
    kind="bar",
    stacked=True,
    width=0.5,
    color=plot_colors,
    linewidth=0,
    ax=ax
)

ax.set_title("Cell Subsets at T–B Border", fontsize=11)
ax.set_xlabel("Day", fontsize=10)
ax.set_ylabel("Cell count", fontsize=10)

ax.set_xticklabels([f"d{day}" for day in day_order], rotation=0)
ax.tick_params(labelsize=9)
ax.margins(x=0.06)          # less space left/right of the outer bars

ax.legend(
    title="Tg subtype",
    bbox_to_anchor=(1.0, 1),
    loc="upper left",
    frameon=False,
    fontsize=8,
    title_fontsize=9,
    handlelength=1,
    borderaxespad=0.2
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)
plt.tight_layout()

fig.savefig(
    "Tg_cell_subsets_TB_border_stacked.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
    dpi=300
)

plt.show()

### Fraction of Tg cells located at the border → Figure 4e

Per section: Tg cells in the border corridor / all Tg cells of that subset in the section.
Bars per subset and day (`Fig4e_...`) are shown in the manuscript; the other variants (all subsets pooled,
box plot, all Tg cells) are not.

In [ ]:
# ============================================================
# Fraction of each Tg subset located at the T–B border
#
#   fraction = (Tg cells of subtype X within ±150 µm of a QC-passed border)
#              / (ALL Tg cells of subtype X in the same lymph node, whole tissue)
#
# Computed per lymph node, then mean ± SEM across LNs per day.
# Grouped (not stacked) bars, because each subtype has its own denominator.
#
# Numerator  : tg_plot         (border corridor, QC-passed segments only)
# Denominator: all_results_df  (all cells, main parameter set, whole tissue)
# ============================================================


module_palette = {
    "Tg_pre_Tfh": "#6baed6",
    "Tg_Tfh": "#08519c",
    "Tg_pre_Th1": "#fdae6b",
    "Tg_Th1": "#a63603",
}
tg_types = ["Tg_pre_Th1", "Tg_Th1", "Tg_pre_Tfh", "Tg_Tfh"]
short = {"Tg_pre_Th1": "pre-Th1", "Tg_Th1": "Th1", "Tg_pre_Tfh": "pre-Tfh", "Tg_Tfh": "Tfh"}
day_order = [2, 3, 4, 6]
MIN_TG_PER_LN = 10          # LNs with fewer Tg cells of a subtype are skipped for that subtype
FS = 8


def add_ids(df):
    df = df.copy()
    df["LN"] = df["LN"].astype(str)
    df["day"] = df["LN"].str.extract(r"Day(\d+)", expand=False).astype(float)
    df["LN_uid"] = (df["run"].astype(str) + "|" + df["LN"]) if "run" in df.columns else df["LN"]
    df["fine_type"] = df["fine_type"].astype(str)
    return df[df["fine_type"].isin(tg_types) & df["day"].isin(day_order)]


# ------------------------------------------------------------
# 1. Counts per LN x subtype
# ------------------------------------------------------------
border = add_ids(tg_plot)          # numerator
total = add_ids(all_results_df)    # denominator (whole tissue)

n_border = border.groupby(["LN_uid", "day", "fine_type"]).size().rename("n_border")
n_total = total.groupby(["LN_uid", "day", "fine_type"]).size().rename("n_total")

frac_ln = pd.concat([n_border, n_total], axis=1).fillna(0).reset_index()
frac_ln = frac_ln[frac_ln["n_total"] >= MIN_TG_PER_LN].copy()
frac_ln["pct_at_border"] = 100 * frac_ln["n_border"] / frac_ln["n_total"]

# sanity check: border cells can never exceed all cells
assert (frac_ln["n_border"] <= frac_ln["n_total"]).all(), "border count > total count – check LN/run ids"

summ = (
    frac_ln.groupby(["day", "fine_type"])["pct_at_border"]
    .agg(mean="mean", sd="std", n_LN="size")
    .reset_index()
)
summ["sem"] = summ["sd"] / np.sqrt(summ["n_LN"])

display(summ.pivot(index="day", columns="fine_type", values="mean").reindex(columns=tg_types).round(1))
display(frac_ln.sort_values(["day", "fine_type", "LN_uid"]))

# ------------------------------------------------------------
# 2. Grouped barplot: mean ± SEM, dots = lymph nodes
# ------------------------------------------------------------
bar_w = 0.19
offsets = dict(zip(tg_types, (np.arange(len(tg_types)) - (len(tg_types) - 1) / 2) * bar_w))
x_of_day = {d: i for i, d in enumerate(day_order)}
rng = np.random.default_rng(0)

with plt.rc_context({"font.size": FS, "axes.linewidth": 0.7}):
    fig, ax = plt.subplots(figsize=(3.6, 2.8))

    for tg in tg_types:
        c = module_palette[tg]
        s = summ[summ["fine_type"] == tg]
        xs = s["day"].map(x_of_day) + offsets[tg]
        ax.bar(xs, s["mean"], width=bar_w * 0.92, color=c, linewidth=0, label=short[tg], zorder=2)
        ax.errorbar(xs, s["mean"], yerr=s["sem"].fillna(0), fmt="none",
                    ecolor="black", elinewidth=0.7, capsize=1.5, zorder=3)

        p = frac_ln[frac_ln["fine_type"] == tg]
        px = p["day"].map(x_of_day) + offsets[tg] + rng.uniform(-0.04, 0.04, len(p))
        ax.scatter(px, p["pct_at_border"], s=7, color="white", edgecolors="black",
                   linewidths=0.4, zorder=4)

    ax.set_xticks(range(len(day_order)))
    ax.set_xticklabels([f"d{d}" for d in day_order])
    ax.set_xlabel("Day")
    ax.set_ylabel("Tg cells at T–B border\n(% of all cells of subset)")
    ax.set_ylim(0, None)
    ax.legend(title="Tg subset", bbox_to_anchor=(1.0, 1), loc="upper left", frameon=False,
              fontsize=FS - 1, title_fontsize=FS, handlelength=1, borderaxespad=0.2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=2.5, width=0.7)
    plt.tight_layout()

    fig.savefig("Tg_fraction_at_TB_border_of_all_Tg_per_subset.pdf",
                bbox_inches="tight", pad_inches=0.02, dpi=300)
    plt.show()

frac_ln.to_csv("Tg_fraction_at_TB_border_per_LN.csv", index=False)

# ------------------------------------------------------------
# 2a. Grouped barplot: mean ± SEM, no dots
# ------------------------------------------------------------
bar_w = 0.19
offsets = dict(zip(tg_types, (np.arange(len(tg_types)) - (len(tg_types) - 1) / 2) * bar_w))
x_of_day = {d: i for i, d in enumerate(day_order)}
YLABEL = "Tg cells at T–B border\n(% of all Tg cells of subset in LN)"

with plt.rc_context({"font.size": FS, "axes.linewidth": 0.7}):
    fig, ax = plt.subplots(figsize=(3.6, 2.8))

    for tg in tg_types:
        s = summ[summ["fine_type"] == tg]
        xs = s["day"].map(x_of_day) + offsets[tg]
        ax.bar(xs, s["mean"], width=bar_w * 0.92, color=module_palette[tg],
               linewidth=0, label=short[tg], zorder=2)
        ax.errorbar(xs, s["mean"], yerr=s["sem"].fillna(0), fmt="none",
                    ecolor="black", elinewidth=0.7, capsize=1.5, zorder=3)

    ax.set_xticks(range(len(day_order)))
    ax.set_xticklabels([f"d{d}" for d in day_order])
    ax.set_xlabel("Day")
    ax.set_ylabel(YLABEL)
    ax.set_ylim(0, None)
    ax.legend(title="Tg subset", bbox_to_anchor=(1.0, 1), loc="upper left", frameon=False,
              fontsize=FS - 1, title_fontsize=FS, handlelength=1, borderaxespad=0.2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=2.5, width=0.7)
    plt.tight_layout()

    fig.savefig("Fig4e_Tg_fraction_at_TB_border_per_subset_bars.pdf", bbox_inches="tight", pad_inches=0.02)
    plt.show()


# ------------------------------------------------------------
# 2b. Grouped boxplots (one box per subset and day, based on LN values)
# ------------------------------------------------------------
with plt.rc_context({"font.size": FS, "axes.linewidth": 0.7}):
    fig, ax = plt.subplots(figsize=(3.6, 2.8))

    for tg in tg_types:
        data, pos = [], []
        for d in day_order:
            v = frac_ln.loc[(frac_ln["fine_type"] == tg) & (frac_ln["day"] == d), "pct_at_border"]
            if len(v) == 0:
                continue
            data.append(v.to_numpy())
            pos.append(x_of_day[d] + offsets[tg])

        bp = ax.boxplot(data, positions=pos, widths=bar_w * 0.85, patch_artist=True,
                        showfliers=False, whis=(0, 100),        # whiskers = min/max (few LNs)
                        medianprops=dict(color="black", linewidth=0.9),
                        boxprops=dict(linewidth=0.6),
                        whiskerprops=dict(linewidth=0.6),
                        capprops=dict(linewidth=0.6))
        for box in bp["boxes"]:
            box.set_facecolor(module_palette[tg])
            box.set_edgecolor("black")
        ax.bar([np.nan], [np.nan], color=module_palette[tg], label=short[tg])   # legend handle

    ax.set_xticks(range(len(day_order)))
    ax.set_xticklabels([f"d{d}" for d in day_order])
    ax.set_xlim(-0.5, len(day_order) - 0.5)
    ax.set_xlabel("Day")
    ax.set_ylabel(YLABEL)
    ax.set_ylim(0, None)
    ax.legend(title="Tg subset", bbox_to_anchor=(1.0, 1), loc="upper left", frameon=False,
              fontsize=FS - 1, title_fontsize=FS, handlelength=1, borderaxespad=0.2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=2.5, width=0.7)
    plt.tight_layout()

    fig.savefig("Tg_fraction_at_TB_border_per_subset_boxplot.pdf", bbox_inches="tight", pad_inches=0.02)
    plt.show()

frac_ln.to_csv("Tg_fraction_at_TB_border_per_LN.csv", index=False)

In [ ]:
# ============================================================
# Fraction of Tg cells at the T–B border (all Tg subsets pooled)
#
#   pct_at_border = Tg cells within ±150 µm of a QC-passed border segment
#                   / ALL Tg cells in the same LN (whole tissue)
#
# Per LN, then mean ± SEM across LNs per day; dots = LNs.
# Input: all_results_df (main parameter set, whole tissue)
# ============================================================


mpl.rcParams["pdf.fonttype"] = 42

tg_types = ["Tg_pre_Th1", "Tg_Th1", "Tg_pre_Tfh", "Tg_Tfh"]
day_order = [2, 3, 4, 6]
MIN_TG_PER_LN = 10
FS = 8
TG_COLOR = "#d0021b"

# ------------------------------------------------------------
# 1. Per LN: Tg at border / all Tg
# ------------------------------------------------------------
df = all_results_df.copy()
df["LN"] = df["LN"].astype(str)
df["day"] = df["LN"].str.extract(r"Day(\d+)", expand=False).astype(float)
df["LN_uid"] = df["run"].astype(str) + "|" + df["LN"]
df = df[df["day"].isin(day_order) & df["fine_type"].astype(str).isin(tg_types)]

frac_ln = (df.groupby(["LN_uid", "day"])
             .agg(n_border=("tb_analysis_keep", "sum"), n_total=("tb_analysis_keep", "size"))
             .reset_index())
frac_ln = frac_ln[frac_ln["n_total"] >= MIN_TG_PER_LN].copy()
frac_ln["pct_at_border"] = 100 * frac_ln["n_border"] / frac_ln["n_total"]

summ = (frac_ln.groupby("day")["pct_at_border"]
        .agg(mean="mean", sd="std", n_LN="size").reindex(day_order).reset_index())
summ["sem"] = summ["sd"] / np.sqrt(summ["n_LN"])

display(summ.round(1))
display(frac_ln.sort_values(["day", "LN_uid"]))

# ------------------------------------------------------------
# 2. Plot: mean ± SEM per day, dots = LNs
# ------------------------------------------------------------
x_of_day = {d: i for i, d in enumerate(day_order)}
rng = np.random.default_rng(0)

with plt.rc_context({"font.size": FS, "axes.linewidth": 0.7}):
    fig, ax = plt.subplots(figsize=(2.0, 2.4))

    xs = summ["day"].map(x_of_day)
    ax.bar(xs, summ["mean"], width=0.6, color=TG_COLOR, alpha=0.85, linewidth=0, zorder=2)
    ax.errorbar(xs, summ["mean"], yerr=summ["sem"].fillna(0), fmt="none",
                ecolor="black", elinewidth=0.7, capsize=2, zorder=3)
    ax.scatter(frac_ln["day"].map(x_of_day) + rng.uniform(-0.1, 0.1, len(frac_ln)),
               frac_ln["pct_at_border"], s=9, color="white", edgecolors="black",
               linewidths=0.4, zorder=4)

    ax.set_xticks(range(len(day_order)))
    ax.set_xticklabels([f"d{d}" for d in day_order])
    ax.set_xlabel("Day")
    ax.set_ylabel("Tg cells at T–B border\n(% of all Tg cells in LN)")
    ax.set_ylim(0, None)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=2.5, width=0.7)
    plt.tight_layout()

    fig.savefig("Tg_all_fraction_at_TB_border.pdf", bbox_inches="tight", pad_inches=0.02)
    fig.savefig("Tg_all_fraction_at_TB_border.png", dpi=600, bbox_inches="tight", pad_inches=0.02)
    plt.show()

frac_ln.to_csv("Tg_all_fraction_at_TB_border_per_LN.csv", index=False)

### Occupancy near the interface per run

In [ ]:
def plot_tg_occupancy_near_tb_interface_per_run(
    all_results_df,
    tg_types=("Tg_pre_Th1", "Tg_Th1", "Tg_Tfh", "Tg_pre_Tfh"),
    border_zone_um=150,
    run_order=("run1", "run2"),
    figsize=None
):
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt

    all_cells = all_results_df.copy()

    required = {
        "run",
        "fine_type",
        "dist_to_border",
        "tb_analysis_keep"
    }

    missing = required - set(all_cells.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # --------------------------------------------------------
    # Keep Tg cells only
    # --------------------------------------------------------

    tg_df = all_cells[
        all_cells["fine_type"].isin(tg_types)
    ].copy()

    # --------------------------------------------------------
    # BEFORE QC:
    # raw geometric ± border_zone_um corridor
    # --------------------------------------------------------

    tg_df["inside_raw_border_zone"] = (
        tg_df["dist_to_border"].abs() <= border_zone_um
    )

    # --------------------------------------------------------
    # AFTER QC:
    # QC-passed cells
    # --------------------------------------------------------

    tg_df["inside_qc_border_zone"] = tg_df["tb_analysis_keep"]

    # --------------------------------------------------------
    # Build plotting dataframe
    # --------------------------------------------------------

    plot_records = []

    run_order = [
        run for run in run_order
        if run in tg_df["run"].dropna().unique()
    ]

    for run in run_order:

        tg_df_run = tg_df[tg_df["run"] == run].copy()

        for tg in tg_types:

            sub = tg_df_run[tg_df_run["fine_type"] == tg]

            # Before QC
            n_before_inside = sub["inside_raw_border_zone"].sum()
            n_before_outside = len(sub) - n_before_inside

            # After QC
            n_after_inside = sub["inside_qc_border_zone"].sum()
            n_after_outside = len(sub) - n_after_inside

            plot_records.extend([
                {
                    "run": run,
                    "fine_type": tg,
                    "stage": "Before QC",
                    "zone": "Inside corridor",
                    "n_cells": n_before_inside
                },
                {
                    "run": run,
                    "fine_type": tg,
                    "stage": "Before QC",
                    "zone": "Outside corridor",
                    "n_cells": n_before_outside
                },
                {
                    "run": run,
                    "fine_type": tg,
                    "stage": "After QC",
                    "zone": "Inside corridor",
                    "n_cells": n_after_inside
                },
                {
                    "run": run,
                    "fine_type": tg,
                    "stage": "After QC",
                    "zone": "Outside corridor",
                    "n_cells": n_after_outside
                }
            ])

    plot_df = pd.DataFrame(plot_records)

    plot_df["fine_type"] = pd.Categorical(
        plot_df["fine_type"],
        categories=list(tg_types),
        ordered=True
    )

    plot_df["stage"] = pd.Categorical(
        plot_df["stage"],
        categories=["Before QC", "After QC"],
        ordered=True
    )

    plot_df["zone"] = pd.Categorical(
        plot_df["zone"],
        categories=["Inside corridor", "Outside corridor"],
        ordered=True
    )

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    sns.set_style("whitegrid")

    n_runs = len(run_order)

    if figsize is None:
        figsize = (8, 2.5 * n_runs)

    fig, axes = plt.subplots(
        n_runs,
        2,
        figsize=figsize,
        sharey=False,
        squeeze=False
    )

    grey_palette = {
        "Inside corridor": "#6a6a6a",
        "Outside corridor": "#d0d0d0",
    }

    stage_order = ["Before QC", "After QC"]

    for r, run in enumerate(run_order):

        for c, stage in enumerate(stage_order):

            ax = axes[r, c]

            sub_plot = plot_df[
                (plot_df["run"] == run) &
                (plot_df["stage"] == stage)
            ].copy()

            sns.barplot(
                data=sub_plot,
                x="fine_type",
                y="n_cells",
                hue="zone",
                order=list(tg_types),
                hue_order=["Inside corridor", "Outside corridor"],
                palette=grey_palette,
                ax=ax
            )

            ax.set_title(
                f"{run} | {stage}",
                fontsize=11,
                pad=10
            )

            ax.set_xlabel("Tg subtype", fontsize=13)

            if c == 0:
                ax.set_ylabel("Number of cells", fontsize=13)
            else:
                ax.set_ylabel("")

            ax.tick_params(axis="x", labelsize=10)

            for label in ax.get_xticklabels():
                label.set_rotation(25)
                label.set_ha("right")
                label.set_rotation_mode("anchor")

            ax.tick_params(
                axis="y",
                labelsize=11
            )

            if r == 0 and c == 1:
                ax.legend(
                    title="Zone",
                    frameon=False,
                    bbox_to_anchor=(1.02, 1),
                    loc="upper left"
                )
            else:
                ax.get_legend().remove()

    fig.suptitle(
        f"Tg occupancy near the T/B interface ±{border_zone_um} µm",
        fontsize=17,
        y=1.02
    )

    sns.despine()
    plt.tight_layout()

    return fig, axes, plot_df

In [ ]:
fig_occ, axes_occ, tg_occupancy_df = plot_tg_occupancy_near_tb_interface_per_run(
    all_results_df,
    tg_types=tg_types,
    border_zone_um=150
)

plt.show()

display(tg_occupancy_df)

### Median distance per sample

In [ ]:
def plot_tg_spatial_story(
    summary,
    tg_types=("Tg_pre_Th1", "Tg_Th1", "Tg_Tfh", "Tg_pre_Tfh"),
    border_zone_um=150,
    figsize=(7.8, 5.2),
    run_order=("run1", "run2"),
    run_offsets=None,
    jitter=0.04
):
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    sns.set_style("white")

    summary = summary.copy()

    if run_offsets is None:
        run_offsets = {"run1": -0.16, "run2": 0.16}

    if "day" not in summary.columns:
        raise ValueError("summary needs a 'day' column")
    if "run" not in summary.columns:
        raise ValueError("summary needs a 'run' column")

    # keep only requested Tg types
    summary = summary[summary["fine_type"].isin(tg_types)].copy()

    summary["fine_type"] = pd.Categorical(
        summary["fine_type"],
        categories=list(tg_types),
        ordered=True
    )

    summary["day_label"] = "Day" + summary["day"].astype(int).astype(str)

    day_order = sorted(summary["day"].dropna().astype(int).unique().tolist())
    day_labels = [f"Day{d}" for d in day_order]

    palette = sns.color_palette("cool", n_colors=len(day_labels))
    day_to_color = dict(zip(day_labels, palette))

    fig, ax = plt.subplots(figsize=figsize)

    # --------------------------------------------------------
    # boxplot FIRST and with low zorder -> underneath the dots
    # --------------------------------------------------------
    sns.boxplot(
        data=summary,
        x="fine_type",
        y="median_dist",
        order=list(tg_types),
        showfliers=False,
        width=0.55,
        color="white",
        linewidth=1.2,
        zorder=1,
        boxprops={"facecolor": "white", "alpha": 0.75, "edgecolor": "0.55", "zorder": 1},
        whiskerprops={"color": "0.55", "zorder": 1},
        capprops={"color": "0.55", "zorder": 1},
        medianprops={"color": "0.35", "linewidth": 1.4, "zorder": 2},
        ax=ax
    )

    # --------------------------------------------------------
    # dots SECOND and with higher zorder -> on top of boxplot
    # --------------------------------------------------------
    x_map = {ft: i for i, ft in enumerate(tg_types)}
    rng = np.random.default_rng(42)

    for _, row in summary.iterrows():
        x = x_map[row["fine_type"]]
        x += run_offsets.get(row["run"], 0.0)
        x += rng.uniform(-jitter, jitter)

        ax.scatter(
            x,
            row["median_dist"],
            s=55,
            color=day_to_color[row["day_label"]],
            alpha=0.95,
            edgecolor="white",
            linewidth=0.4,
            zorder=3
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=1, zorder=0)
    ax.axhline(border_zone_um, color="grey", linestyle=":", linewidth=1, zorder=0)
    ax.axhline(-border_zone_um, color="grey", linestyle=":", linewidth=1, zorder=0)

    ax.set_ylabel("Median signed distance to T–B border (µm)", fontsize=11)
    ax.set_xlabel("Tg subtype", fontsize=12)
    ax.set_title("Spatial positioning of Tg subtypes", fontsize=13)
    ax.tick_params(axis="x", rotation=30)

    # legend for day colors
    day_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            markersize=7,
            markerfacecolor=day_to_color[d],
            markeredgecolor="none",
            label=d
        )
        for d in day_labels
    ]

    legend1 = ax.legend(
        handles=day_handles,
        title="Day",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=True
    )
    ax.add_artist(legend1)

    # legend for run position
    run_handles = [
        Line2D([0], [0], marker="o", linestyle="", color="grey", markersize=7, label="run1 = left"),
        Line2D([0], [0], marker="o", linestyle="", color="grey", markersize=7, label="run2 = right"),
    ]

    ax.legend(
        handles=run_handles,
        title="Run position",
        bbox_to_anchor=(1.02, 0.55),
        loc="upper left",
        frameon=True
    )

    plt.tight_layout()
    return fig, ax

In [ ]:
fig1, ax1 = plot_tg_spatial_story(
    tg_summary,
    tg_types=tg_types,
    border_zone_um=150, figsize=(6, 4.5),
)

plt.show()

## 6. Tg density: T/B border vs deep T zone vs deep B zone

Zones per cell: border = `tb_analysis_keep`; deep T = ≥ 300 µm on the T side, deep B = ≥ 200 µm on the
B side, both ≥ 100 µm from the tissue edge. Zone area from 20-µm pixels (majority zone per pixel).
Tg cells per mm² per section and zone; bars = mean ± SEM across sections.

In [ ]:
# ============================================================
# Tg cell density (cells / mm²): T/B border vs deep T zone vs deep B zone
#
# Zones (per cell, from the border inference):
#   Border : within ±150 µm of a QC-passed border segment (tb_analysis_keep)
#   Deep T : dist_to_border >= +DEEP_T_UM, >= EDGE_MIN_UM from tissue edge
#   Deep B : dist_to_border <= -DEEP_B_UM, >= EDGE_MIN_UM from tissue edge
#
# Area per zone: tissue rasterized in PIX_UM pixels; each occupied pixel gets
# the majority zone of its cells; area = n_pixels * PIX_UM².
#
# Output: (1) zone map for EVERY lymph node, (2) barplot per day (no statistics)
# Input : all_results_df
# ============================================================


mpl.rcParams["pdf.fonttype"] = 42

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------
DEEP_T_UM = 300
DEEP_B_UM = 200
EDGE_MIN_UM = 100
PIX_UM = 20
MIN_AREA_MM2 = 0.01

tg_types = ["Tg_pre_Th1", "Tg_Th1", "Tg_pre_Tfh", "Tg_Tfh"]
ZONES = ["Deep B", "Border", "Deep T"]
ZONE_COLORS = {"Deep B": "#4a90e2", "Border": "#555555", "Deep T": "#3fd6c1"}
day_order = [2, 3, 4, 6]
FS = 8

METRIC = "tg_per_mm2"     # or "tg_pct_of_cells"
YLAB = {"tg_per_mm2": "Tg T cells per mm²", "tg_pct_of_cells": "Tg T cells (% of cells)"}[METRIC]


# ------------------------------------------------------------
# 2. ZONE PER CELL
# ------------------------------------------------------------
df = all_results_df.copy()
df["LN"] = df["LN"].astype(str)
df["day"] = df["LN"].str.extract(r"Day(\d+)", expand=False).astype(float)
df["LN_uid"] = df["run"].astype(str) + "|" + df["LN"]
df = df[df["day"].isin(day_order)].copy()
df["is_tg"] = df["fine_type"].astype(str).isin(tg_types)

d = df["dist_to_border"]
edge_ok = df["dist_to_tissue_edge"] >= EDGE_MIN_UM
df["zone"] = None
df.loc[df["tb_analysis_keep"].astype(bool), "zone"] = "Border"
df.loc[(d >= DEEP_T_UM) & edge_ok, "zone"] = "Deep T"
df.loc[(d <= -DEEP_B_UM) & edge_ok, "zone"] = "Deep B"


# ------------------------------------------------------------
# 3. AREA PER ZONE
# ------------------------------------------------------------
pix_area_mm2 = (PIX_UM / 1000) ** 2
df["pi"] = np.floor(df["x"] / PIX_UM).astype(int)
df["pj"] = np.floor(df["y"] / PIX_UM).astype(int)

pix = (df.dropna(subset=["zone"])
         .groupby(["LN_uid", "pi", "pj"])["zone"]
         .agg(lambda z: z.value_counts().idxmax())
         .reset_index())
area = (pix.groupby(["LN_uid", "zone"]).size() * pix_area_mm2).rename("area_mm2")


# ------------------------------------------------------------
# 4. DENSITIES PER LN x ZONE
# ------------------------------------------------------------
counts = (df.dropna(subset=["zone"])
            .groupby(["LN_uid", "day", "zone"])
            .agg(n_tg=("is_tg", "sum"), n_all=("is_tg", "size"))
            .reset_index())
dens = counts.merge(area.reset_index(), on=["LN_uid", "zone"], how="left")
dens = dens[dens["area_mm2"] >= MIN_AREA_MM2].copy()
dens["tg_per_mm2"]      = dens["n_tg"] / dens["area_mm2"]
dens["all_per_mm2"]     = dens["n_all"] / dens["area_mm2"]
dens["tg_pct_of_cells"] = 100 * dens["n_tg"] / dens["n_all"]

display(dens.pivot_table(index=["day", "LN_uid"], columns="zone", values=METRIC).round(1))


# ------------------------------------------------------------
# 5. ZONE MAP FOR EVERY LN (grid, sorted by day)
# ------------------------------------------------------------
ln_list = (df[["LN_uid", "day"]].drop_duplicates()
             .sort_values(["day", "LN_uid"])["LN_uid"].tolist())
ncol = 4
nrow = int(np.ceil(len(ln_list) / ncol))

with plt.rc_context({"font.size": FS}):
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 2.6, nrow * 2.3))
    axes = np.atleast_1d(axes).ravel()

    for ax, ln_uid in zip(axes, ln_list):
        sub = df[df["LN_uid"] == ln_uid]
        ax.scatter(sub["x"], sub["y"], s=0.2, color="#e6e6e6", linewidths=0, rasterized=True)
        for z in ZONES:
            s = sub[sub["zone"] == z]
            ax.scatter(s["x"], s["y"], s=0.3, color=ZONE_COLORS[z], alpha=0.6,
                       linewidths=0, rasterized=True)
        t = sub[sub["is_tg"]]
        ax.scatter(t["x"], t["y"], s=0.8, color="#d0021b", linewidths=0, rasterized=True)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.axis("off")
        ax.set_title(f"d{int(sub['day'].iloc[0])}  {ln_uid.split('|')[1]}", fontsize=FS)

    for ax in axes[len(ln_list):]:
        ax.axis("off")

    handles = [plt.Line2D([], [], marker="o", ls="", color=ZONE_COLORS[z], markersize=5) for z in ZONES] + \
              [plt.Line2D([], [], marker="o", ls="", color="#d0021b", markersize=5)]
    fig.legend(handles, ZONES + ["Tg T cells"], loc="lower center", ncol=4, frameon=False,
               fontsize=FS, bbox_to_anchor=(0.5, -0.01))
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig("Zones_QC_all_LN.pdf", bbox_inches="tight", dpi=300)
    plt.show()


# ------------------------------------------------------------
# 6. BARPLOT per day (mean ± SEM across sections, dots = sections)
# ------------------------------------------------------------
x_of_day = {dd: i for i, dd in enumerate(day_order)}
w_bar = 0.26
off = {z: (n - 1) * w_bar for n, z in enumerate(ZONES)}

summ = (dens.groupby(["day", "zone"])[METRIC]
            .agg(mean="mean", sd="std", n="size").reset_index())
summ["sem"] = summ["sd"] / np.sqrt(summ["n"])

with plt.rc_context({"font.size": FS, "axes.linewidth": 0.7}):
    fig, ax = plt.subplots(figsize=(3.6, 2.6))

    for z in ZONES:
        s = summ[summ["zone"] == z]
        xs = s["day"].map(x_of_day) + off[z]
        ax.bar(xs, s["mean"], width=w_bar * 0.92, color=ZONE_COLORS[z], alpha=0.85,
               linewidth=0, label=z, zorder=2)
        ax.errorbar(xs, s["mean"], yerr=s["sem"].fillna(0), fmt="none",
                    ecolor="black", elinewidth=0.6, capsize=1.5, zorder=3)

    rng = np.random.default_rng(0)
    for z in ZONES:
        s = dens[dens["zone"] == z]
        xs = s["day"].map(x_of_day) + off[z] + rng.uniform(-0.05, 0.05, len(s))
        ax.scatter(xs, s[METRIC], s=6, color="white", edgecolors="black",
                   linewidths=0.4, zorder=5)

    ax.set_xticks(range(len(day_order)))
    ax.set_xticklabels([f"d{dd}" for dd in day_order])
    ax.set_xlabel("Day")
    ax.set_ylabel(YLAB)
    ax.set_ylim(0, None)
    ax.legend(frameon=False, fontsize=FS - 1, loc="upper left", bbox_to_anchor=(1, 1), handlelength=1)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    fig.savefig(f"Tg_density_border_vs_deep_zones_{METRIC}.pdf", bbox_inches="tight", pad_inches=0.02)
    plt.show()

dens.to_csv("Tg_density_per_zone_per_LN.csv", index=False)

## 7. ECDF of the signed distance to the border → Figure 4f

Per Tg subtype × day: ECDF computed for every section (≥ 10 cells, QC-passed segments only);
curve = mean across sections, band = ± SEM. Day 0 not shown.

Plotting function (2 × 2 layout, fixed day colours).

In [ ]:
mpl.rcParams["pdf.fonttype"] = 42        # editable text in PDF
mpl.rcParams["ps.fonttype"] = 42
# mpl.rcParams["font.family"] = "Arial"  # uncomment if Arial is installed


def plot_tg_ecdf_from_curves(
    ecdf_curve_df,
    tg_types=("Tg_pre_Th1", "Tg_Th1", "Tg_pre_Tfh", "Tg_Tfh"),
    border_zone_um=150,
    ncols=2,                     # 2 -> 2x2 grid, 4 -> one row
    figsize=(2.6, 2.8),          # inches = final print size (~6.6 x 7.1 cm)
    fs_tick=7,
    fs_label=8,
    fs_title=8,
    fs_text=6.5,
    fs_legend=7,
    lw_curve=1.1,
    lw_axes=0.6,
    zone_text_panel="Tg_Th1",    # panel with "B side"/"T side"; None = all panels
    all_days=(0, 2, 3, 4, 6),    # fixed palette over ALL days -> same colors in every figure
    exclude_days=(0,),           # days not plotted (colors of the others stay fixed)
):
    titles = {"Tg_pre_Th1": "pre-Th1", "Tg_Th1": "Th1", "Tg_pre_Tfh": "pre-Tfh", "Tg_Tfh": "Tfh"}

    # fixed palette: d0 keeps its slot, so d2 ... d6 always get the same color
    day_palette = dict(zip(all_days, sns.color_palette("cool", n_colors=len(all_days))))
    present = set(ecdf_curve_df["day"].dropna().unique())
    day_order = [d for d in all_days if d in present and d not in exclude_days]

    nrows = int(np.ceil(len(tg_types) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    for ax, tg in zip(axes, tg_types):
        sub_tg = ecdf_curve_df[ecdf_curve_df["fine_type"] == tg]

        for day in day_order:
            d = sub_tg[sub_tg["day"] == day]
            if d.empty:
                continue
            col = day_palette[day]
            ax.fill_between(d["distance"], d["ecdf_low"], d["ecdf_high"],
                            color=col, alpha=0.18, linewidth=0)
            ax.plot(d["distance"], d["ecdf_mean"], color=col, linewidth=lw_curve,
                    label=f"d{int(day)}")

        ax.axvline(0, color="black", linestyle="--", linewidth=lw_axes)
        ax.axvline(-border_zone_um, color="grey", linestyle=":", linewidth=lw_axes)
        ax.axvline(border_zone_um, color="grey", linestyle=":", linewidth=lw_axes)

        if zone_text_panel is None or tg == zone_text_panel:
            ax.text(0.03, 0.97, "B side", transform=ax.transAxes, ha="left", va="top",
                    fontsize=fs_text, color="#555")
            ax.text(0.97, 0.03, "T side", transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=fs_text, color="#555")

        ax.set_title(titles.get(tg, tg), fontsize=fs_title, pad=2)
        ax.set_xlim(-border_zone_um * 1.05, border_zone_um * 1.05)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xticks([-100, 0, 100])
        ax.set_yticks([0, 0.5, 1])
        ax.set_yticklabels(["0", "0.5", "1"])
        ax.tick_params(labelsize=fs_tick, length=2, width=lw_axes, pad=1.5)
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)
        for side in ["left", "bottom"]:
            ax.spines[side].set_linewidth(lw_axes)

    for ax in axes[len(tg_types):]:
        ax.axis("off")

    fig.supxlabel("Distance to T/B border (µm)", fontsize=fs_label, y=0.07)
    fig.supylabel("Cumulative fraction", fontsize=fs_label, x=0.01)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.03),
               ncol=len(labels), frameon=False, fontsize=fs_legend,
               handlelength=1.0, columnspacing=0.6, handletextpad=0.3)

    fig.tight_layout(rect=(0.02, 0.08, 1, 1), h_pad=0.4, w_pad=0.3)

    return fig, axes

In [ ]:
# ============================================================
# ECDF of Tg subsets relative to the T/B border
# - only cells assigned to a QC-passed (valid) border segment
# - variability ACROSS LYMPH NODES (biological replicates)
#
# For every Tg subtype x day:
#   1. keep only cells with a valid nearest border segment
#      (same rule as the old bootstrap: cells without a segment
#       were never sampled there)
#   2. ECDF of signed distance computed separately for each LN
#   3. curve = mean of the per-LN ECDFs (each LN weighted equally)
#      band  = mean ± SEM across LNs  (or "sd" / "range")
#   4. ECDF-AUC per LN returned for later plotting (not plotted here)
#
# Sign convention: negative distance = B-zone side, positive = T-zone side
# Required columns in tg_plot: fine_type, dist_to_border, day, LN,
#                              nearest_border_segment (+ run if LN ids repeat)
# ============================================================


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
BORDER_UM = 150
MIN_CELLS_PER_LN = 10
BAND = "sem"                         # "sem", "sd" or "range"

SEGMENT_COL = "nearest_border_segment"

# Optional extra QC column (set to None if not needed).
# Examples: "segment_reliable_final" with True,
#           "tb_region_confident" with values != "excluded"
VALID_COL = None
VALID_VALUES = [True]                # rows with VALID_COL in these values are kept


# ------------------------------------------------------------
# 0. Keep only cells at valid (QC-passed) border segments
# ------------------------------------------------------------
def filter_valid_segments(tg_plot, segment_col=SEGMENT_COL, valid_col=VALID_COL, valid_values=VALID_VALUES):
    n0 = len(tg_plot)
    df = tg_plot.copy()

    # rule 1 (as in the old bootstrap): a valid nearest segment must be assigned
    df = df[df[segment_col].notna()]
    print(f"Cells with an assigned border segment: {len(df)} / {n0}")

    # rule 2 (optional): explicit QC flag
    if valid_col is not None:
        if valid_col not in df.columns:
            raise ValueError(f"'{valid_col}' not in tg_plot. Available: {df.columns.tolist()}")
        print(f"\n{valid_col} before filtering:")
        print(df[valid_col].value_counts(dropna=False))
        df = df[df[valid_col].isin(valid_values)]
        print(f"Cells after QC flag filter: {len(df)}")

    return df


# ------------------------------------------------------------
# 1. Per-LN ECDFs
# ------------------------------------------------------------
def compute_tg_ecdf_per_ln(
    df_valid,
    tg_types=("Tg_pre_Th1", "Tg_Th1", "Tg_pre_Tfh", "Tg_Tfh"),
    distance_col="dist_to_border",
    day_col="day",
    ln_col="LN",
    run_col="run",
    border_zone_um=BORDER_UM,
    min_cells_per_ln=MIN_CELLS_PER_LN,
    band=BAND,
    grid=None,
):
    if grid is None:
        grid = np.linspace(-border_zone_um, border_zone_um, 301)

    df = df_valid[df_valid["fine_type"].isin(tg_types)].copy()
    df = df[df[distance_col].between(-border_zone_um, border_zone_um)]

    # unique LN id across runs
    if run_col in df.columns:
        df["LN_uid"] = df[run_col].astype(str) + "|" + df[ln_col].astype(str)
    else:
        df["LN_uid"] = df[ln_col].astype(str)

    def ecdf_on_grid(x):
        x = np.sort(np.asarray(x))
        return np.searchsorted(x, grid, side="right") / len(x)

    def norm_auc(ecdf_values):
        return np.trapz(ecdf_values, grid) / (grid.max() - grid.min())

    ecdf_records, auc_ln_records = [], []

    for (tg, day), sub in df.groupby(["fine_type", day_col]):
        per_ln = []
        for ln, sub_ln in sub.groupby("LN_uid"):
            vals = sub_ln[distance_col].dropna().values
            if len(vals) < min_cells_per_ln:
                continue
            e = ecdf_on_grid(vals)
            per_ln.append(e)
            auc_ln_records.append({
                "fine_type": tg, "day": day, "LN": ln,
                "auc": norm_auc(e),
                "frac_B_side": (vals <= 0).mean(),     # = ECDF value at the border
                "median_dist_um": np.median(vals),
                "n_cells": len(vals),
                "n_segments": sub_ln[SEGMENT_COL].nunique(),
            })

        if not per_ln:
            continue

        per_ln = np.vstack(per_ln)
        n_ln = per_ln.shape[0]
        mean = per_ln.mean(axis=0)

        if n_ln == 1:
            low = high = mean
        elif band == "sd":
            sd = per_ln.std(axis=0, ddof=1)
            low, high = mean - sd, mean + sd
        elif band == "range":
            low, high = per_ln.min(axis=0), per_ln.max(axis=0)
        else:  # sem
            sem = per_ln.std(axis=0, ddof=1) / np.sqrt(n_ln)
            low, high = mean - sem, mean + sem

        ecdf_records.append(pd.DataFrame({
            "distance": grid,
            "ecdf_mean": mean,
            "ecdf_low": np.clip(low, 0, 1),
            "ecdf_high": np.clip(high, 0, 1),
            "fine_type": tg, "day": day, "n_LN": n_ln,
        }))

    ecdf_curve_df = pd.concat(ecdf_records, ignore_index=True) if ecdf_records else pd.DataFrame()
    auc_ln_df = pd.DataFrame(auc_ln_records)

    auc_df = (
        auc_ln_df.groupby(["fine_type", "day"])
        .agg(auc_mean=("auc", "mean"),
             auc_sd=("auc", lambda v: v.std(ddof=1) if len(v) > 1 else np.nan),
             frac_B_mean=("frac_B_side", "mean"),
             median_dist_mean=("median_dist_um", "mean"),
             n_LN=("auc", "size"),
             n_cells=("n_cells", "sum"))
        .reset_index()
    )
    auc_df["auc_sem"] = auc_df["auc_sd"] / np.sqrt(auc_df["n_LN"])

    return ecdf_curve_df, auc_ln_df, auc_df


# ------------------------------------------------------------
# 2. Run (new names -> old bootstrap results stay untouched)
# ------------------------------------------------------------
tg_plot_valid = filter_valid_segments(tg_plot)

ecdf_curve_df_ln, auc_ln_df, auc_df_ln = compute_tg_ecdf_per_ln(
    tg_plot_valid,
    tg_types=tg_types,
)

# how many LN contribute per subtype x day
print("\nLymph nodes per subtype x day:")
display(auc_df_ln.pivot(index="fine_type", columns="day", values="n_LN"))


# ------------------------------------------------------------
# 3. ECDF plot (reuses plot_tg_ecdf_from_curves)
# ------------------------------------------------------------
fig_ecdf_ln, axes_ecdf_ln = plot_tg_ecdf_from_curves(ecdf_curve_df_ln, tg_types=tg_types)
fig_ecdf_ln.savefig("Fig4f_Tg_ECDF_per_LN_mean_sem.pdf", bbox_inches="tight")
plt.show()

# per-LN values for the revised summary plot later
auc_ln_df.to_csv("Fig4f_ECDF_per_LN_summary.csv", index=False)

### Position summary per section → Figure 4g (`METRIC = "median"`)

One value per section derived from its ECDF (`METRIC`: median distance, mean distance or % on the B side);
dots = sections, error bars = mean ± SEM.

In [ ]:
# ============================================================
# Replacement for the ECDF-AUC plot
# One value per lymph node, derived from the per-LN ECDF:
#
#   METRIC = "median"  -> median position relative to the T–B border (µm)
#                         = distance where the ECDF crosses 0.5
#                         sign flipped: positive = into the B-cell zone
#   METRIC = "mean"    -> mean position relative to the T–B border (µm)
#                         = old ECDF-AUC rescaled to µm (mean = 300*AUC - 150)
#   METRIC = "frac_B"  -> % of Tg cells on the B-zone side
#                         = ECDF value at the border (0 µm)
#
# Dots = individual lymph nodes, error bars = mean ± SEM across LNs.
# Optional second panel: change relative to day 2.
#
# Requires: auc_ln_df  (from tg_ecdf_per_LN_validsegments chunk;
#           columns fine_type, day, LN, frac_B_side, median_dist_um)
#           tg_types, module_palette
# ============================================================


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
METRIC = "median"                  # "median", "mean" or "frac_B"
SHOW_DELTA = False                 # True = add panel "change vs. d2"
REF_DAY = 2
EXCLUDE = [("Tg_pre_Th1", 0), ("Tg_Th1", 0), ("Tg_pre_Tfh", 0), ("Tg_Tfh", 0)]   # all subtypes start at d2
FS = 8                             # font size at final size

short = {"Tg_pre_Th1": "pre-Th1", "Tg_Th1": "Th1", "Tg_pre_Tfh": "pre-Tfh", "Tg_Tfh": "Tfh"}
label_nudge = {}                   # e.g. {"Tg_Th1": -5, "Tg_pre_Th1": 5} to separate end labels

B_COL, T_COL = "#eaf1fb", "#eafaf6"            # background: B zone (blue), T zone (cyan)
B_TXT, T_TXT = "#3a6fb5", "#139c8a"

# ------------------------------------------------------------
# 1. Per-LN values
# ------------------------------------------------------------
vals = auc_ln_df.copy()
vals["fine_type"] = vals["fine_type"].astype(str)
for tg_ex, d_ex in EXCLUDE:
    vals = vals[~((vals["fine_type"] == tg_ex) & (vals["day"] == d_ex))]

if METRIC == "median":
    vals["value"] = -vals["median_dist_um"]           # positive = B-zone side
    ylabel = "Median position relative\nto T–B border (µm)"
    mid, delta_label = 0, "Change vs. d2 (µm)"
elif METRIC == "mean":
    vals["value"] = 300 * vals["auc"] - 150            # = -mean distance, positive = B-zone side
    ylabel = "Mean position relative\nto T–B border (µm)"
    mid, delta_label = 0, "Change vs. d2 (µm)"
else:
    vals["value"] = 100 * vals["frac_B_side"]
    ylabel = "Tg cells on\nB-zone side (%)"
    mid, delta_label = 50, "Change vs. d2\n(percentage points)"

summ = (
    vals.groupby(["fine_type", "day"])["value"]
    .agg(mean="mean", sd=lambda v: v.std(ddof=1) if len(v) > 1 else np.nan, n_LN="size")
    .reset_index()
)
summ["sem"] = summ["sd"] / np.sqrt(summ["n_LN"])

# change vs. reference day (per subtype, relative to the day-2 mean)
ref = summ[summ["day"] == REF_DAY].set_index("fine_type")["mean"]
vals["delta"] = vals["value"] - vals["fine_type"].map(ref)
summ["delta"] = summ["mean"] - summ["fine_type"].map(ref)

display(summ.round(2))

# ------------------------------------------------------------
# 2. Plot
# ------------------------------------------------------------
days = sorted(summ["day"].unique())
types_present = [t for t in tg_types if t in set(summ["fine_type"])]
offsets = dict(zip(types_present, np.linspace(-0.24, 0.24, len(types_present))))
rng = np.random.default_rng(0)

panels = [("value", "mean", ylabel, mid)]
if SHOW_DELTA:
    panels.append(("delta", "delta", delta_label, 0))

with plt.rc_context({"font.size": FS, "axes.linewidth": 0.7}):
    fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 2.4),
                             squeeze=False, gridspec_kw={"wspace": 0.6})
    axes = axes[0]

    for ax, (ycol, mcol, ylab, center) in zip(axes, panels):
        # symmetric y-limits around the center line
        lo = min(vals[ycol].min(), (summ[mcol] - summ["sem"].fillna(0)).min())
        hi = max(vals[ycol].max(), (summ[mcol] + summ["sem"].fillna(0)).max())
        span = max(abs(hi - center), abs(center - lo)) * 1.15
        if METRIC == "frac_B" and ycol == "value":
            ylim = (0, 100)
        else:
            ylim = (center - span, center + span)

        ax.axhspan(center, ylim[1], color=B_COL, zorder=0)
        ax.axhspan(ylim[0], center, color=T_COL, zorder=0)
        ax.axhline(center, color="#777", lw=0.8, ls="--", zorder=1)

        for tg in types_present:
            c = module_palette.get(tg, None)
            p = vals[vals["fine_type"] == tg]
            s = summ[summ["fine_type"] == tg].sort_values("day")

            ax.scatter(p["day"] + offsets[tg] + rng.uniform(-0.04, 0.04, len(p)), p[ycol],
                       s=9, color=c, alpha=0.45, linewidths=0, zorder=2)
            yerr = s["sem"].fillna(0)
            if ycol == "delta":
                yerr = yerr.where(s["day"] != REF_DAY, 0)
            ax.errorbar(s["day"] + offsets[tg], s[mcol], yerr=yerr,
                        color=c, marker="o", markersize=4, linewidth=1.5,
                        capsize=2, elinewidth=0.9, zorder=3)

            last = s.iloc[-1]
            ax.text(max(days) + 0.55, last[mcol] + label_nudge.get(tg, 0), short.get(tg, tg),
                    color=c, fontsize=FS - 0.5, fontweight="bold", va="center")

        ax.set_ylim(*ylim)
        ax.set_xlim(min(days) - 0.6, max(days) + 1.6)
        ax.set_xticks(days)
        ax.set_xticklabels([f"d{int(d)}" for d in days])
        ax.set_xlabel("Day")
        ax.set_ylabel(ylab)
        ax.text(0.02, 0.97, "B-zone", transform=ax.transAxes, ha="left", va="top",
                fontsize=FS - 0.5, color=B_TXT, fontweight="bold")
        ax.text(0.02, 0.03, "T-zone", transform=ax.transAxes, ha="left", va="bottom",
                fontsize=FS - 0.5, color=T_TXT, fontweight="bold")
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.tick_params(length=2.5, width=0.7)

    fname = f"Fig4g_Tg_border_position_{METRIC}{'_with_delta' if SHOW_DELTA else ''}_per_LN.pdf"
    fig.savefig(fname, bbox_inches="tight", pad_inches=0.02)
    plt.show()

vals.to_csv(f"Fig4g_Tg_border_position_{METRIC}_per_LN.csv", index=False)